# 1. Introducción y objetivo del proyecto

## Contexto

Wanderbricks es un marketplace de alquiler vacacional que integra información relacionada con usuarios, anfitriones, propiedades, reservas, pagos, reseñas y actividad de navegación. Debido a la variedad y volumen de estos datos, el proyecto requiere una arquitectura que permita almacenar, transformar y analizar la información de manera eficiente.

En esta Evidencia de Aprendizaje se diseña e implementa una base de datos analítica sobre una arquitectura **Lakehouse**, utilizando **Databricks y Delta Lake**. El propósito es organizar los datos en diferentes capas de procesamiento y generar información confiable para apoyar el análisis y la toma de decisiones del negocio.

## Objetivo general

Diseñar e implementar una base de datos analítica para Wanderbricks mediante una arquitectura Lakehouse, realizando la ingestión, transformación, validación y análisis de los datos, y justificando la elección de este modelo frente a los paradigmas relacional y NoSQL.

## Objetivos específicos

* Explorar y caracterizar las fuentes de datos de Wanderbricks, considerando su volumen, variedad, tipos de datos, calidad y relaciones.
* Implementar una arquitectura de datos organizada en capas **Bronze, Silver y Gold**.
* Utilizar tablas **Delta** para garantizar características como atomicidad, historial de cambios y evolución de esquema.
* Procesar datos semiestructurados mediante estructuras anidadas presentes en el conjunto de datos de navegación.
* Construir consultas analíticas en **SQL y PySpark** que respondan preguntas relevantes para el negocio.
* Comparar los paradigmas relacional, NoSQL y Lakehouse, justificando la arquitectura seleccionada a partir de las características reales de los datos.


# 2. Exploración y descripción de los datos

En esta sección se realiza la exploración del conjunto de datos de Wanderbricks con el propósito de identificar su volumen, variedad, estructura, calidad e interrelaciones. Estos elementos permiten comprender las características del problema y sirven como fundamento para la selección de la arquitectura de datos presentada posteriormente.


## 2.1 Volumen y variedad de los datos

El conjunto de datos de Wanderbricks presenta diferentes entidades relacionadas con la operación del marketplace. La información incluye datos de usuarios, anfitriones, propiedades, destinos, reservas, pagos, reseñas y actividad de navegación.

Esta variedad permite analizar tanto procesos transaccionales como comportamiento de los usuarios y características de las propiedades. Además, el conjunto incluye información estructurada y datos semiestructurados, especialmente en los eventos de navegación, donde existen atributos agrupados mediante estructuras anidadas.

La exploración se realizó inicialmente sobre las tablas disponibles en el catálogo de origen y posteriormente sobre las capas Bronze y Silver construidas en el Lakehouse.


### 2.1.1 Identificación de las tablas de origen

Como primer paso se consultó el catálogo de origen `samples.wanderbricks` para identificar las entidades disponibles y establecer el punto de partida de la exploración.

La consulta permite verificar las tablas proporcionadas por Wanderbricks antes de realizar los procesos de ingestión y transformación en las capas del Lakehouse.


In [0]:
%sql

SHOW TABLES IN samples.wanderbricks;

database,tableName,isTemporary
wanderbricks,amenities,false
wanderbricks,booking_updates,false
wanderbricks,bookings,false
wanderbricks,clickstream,false
wanderbricks,countries,false
wanderbricks,customer_support_logs,false
wanderbricks,destinations,false
wanderbricks,employees,false
wanderbricks,hosts,false
wanderbricks,page_views,false


### 2.1.2 Volumen de las tablas

Una vez identificadas las tablas disponibles, se calculó el número de registros de cada entidad mediante `COUNT(*)`. Se utilizó `UNION ALL` para consolidar los resultados en una única consulta y facilitar la comparación del volumen entre las diferentes entidades.

Esta medición permite identificar cuáles tablas concentran la mayor cantidad de información y proporciona una primera aproximación a la escala del conjunto de datos, aspecto relevante para seleccionar una arquitectura orientada al procesamiento analítico.


In [0]:
%sql

SELECT
    'users' AS tabla,
    COUNT(*) AS registros
FROM samples.wanderbricks.users

UNION ALL

SELECT
    'hosts',
    COUNT(*)
FROM samples.wanderbricks.hosts

UNION ALL

SELECT
    'properties',
    COUNT(*)
FROM samples.wanderbricks.properties

UNION ALL

SELECT
    'bookings',
    COUNT(*)
FROM samples.wanderbricks.bookings

UNION ALL

SELECT
    'payments',
    COUNT(*)
FROM samples.wanderbricks.payments

UNION ALL

SELECT
    'reviews',
    COUNT(*)
FROM samples.wanderbricks.reviews

UNION ALL

SELECT
    'clickstream',
    COUNT(*)
FROM samples.wanderbricks.clickstream

UNION ALL

SELECT
    'page_views',
    COUNT(*)
FROM samples.wanderbricks.page_views

UNION ALL

SELECT
    'destinations',
    COUNT(*)
FROM samples.wanderbricks.destinations

UNION ALL

SELECT
    'countries',
    COUNT(*)
FROM samples.wanderbricks.countries;

tabla,registros
users,124509
hosts,19384
properties,18163
bookings,72247
payments,49638
reviews,99793
clickstream,100000
page_views,500000
destinations,42
countries,168


### 2.2 Estructura y tipos de datos

Para complementar la medición del volumen, se examinó la estructura de las tablas mediante `DESCRIBE`. Se tomó como referencia la tabla `bookings`, debido a que contiene información relacionada con las reservas y constituye una de las entidades transaccionales principales del marketplace.

La consulta permite identificar los nombres de las columnas y sus tipos de datos, lo que facilita determinar cómo está estructurada la información y qué características deben considerarse durante los procesos de transformación y modelado.

La revisión de los tipos de datos también permite diferenciar información numérica, textual, temporal y otros atributos utilizados posteriormente en las consultas analíticas.


In [0]:
%sql

DESCRIBE samples.wanderbricks.bookings;

col_name,data_type,comment
booking_id,bigint,Unique identifier of the booking
user_id,bigint,The identifier of the user making the booking
property_id,bigint,Foreign Key to 'properties'
check_in,date,Check-in date
check_out,date,Check-out date
guests_count,int,Number of guests for booking
total_amount,float,Total amount paid
status,string,"Status of the booking (e.g., pending, confirmed)"
created_at,timestamp,Date the booking was created
updated_at,timestamp,Date the booking was last updated


**Interpretación:** La tabla `bookings` presenta una estructura principalmente tabular y combina diferentes tipos de datos. Los identificadores `booking_id`, `user_id` y `property_id` permiten establecer relaciones entre entidades; las fechas de entrada y salida representan información temporal; `guests_count` y `total_amount` contienen valores numéricos; `status` representa el estado de la reserva; y `created_at` y `updated_at` permiten registrar la evolución temporal de cada registro.

Esta combinación evidencia que el conjunto de datos contiene información transaccional que requiere tanto integridad en las relaciones como capacidades de análisis sobre atributos numéricos, categóricos y temporales.


### 2.2.1 Estructura de los datos de pagos

Además de la información asociada a las reservas, se examinó la estructura de la tabla `payments`, que registra las transacciones económicas realizadas en el marketplace.

La revisión de esta tabla permite identificar los atributos relacionados con los métodos de pago, estados de las transacciones y valores monetarios. Esta información es especialmente relevante para los análisis posteriores de ingresos, transacciones completadas, fallidas y reembolsadas.


In [0]:
%sql

DESCRIBE samples.wanderbricks.payments;

col_name,data_type,comment
payment_id,bigint,Uniquely identifies this payment
booking_id,bigint,The booking that was paid for
amount,"decimal(15,4)",The amount paid
payment_method,string,"The method used for payment (e.g., 'credit_card', 'paypal', 'apple_pay', 'google_pay', 'bank_transfer')"
status,string,"The status of the payment (e.g., 'pending', 'completed', 'failed', 'refunded')"
payment_date,timestamp,The date the payment was made


**Interpretación:** La tabla `payments` contiene información financiera asociada a las reservas. `payment_id` identifica cada pago y `booking_id` establece la relación con la reserva correspondiente. El atributo `amount` utiliza el tipo `decimal(15,4)`, apropiado para representar valores monetarios con precisión. Además, `payment_method` y `status` permiten clasificar las transacciones según el medio utilizado y su estado, mientras que `payment_date` incorpora la dimensión temporal de la operación.

La relación entre `bookings` y `payments` evidencia que los datos no solo contienen atributos individuales, sino también vínculos entre entidades que permiten realizar análisis integrados sobre reservas, transacciones e ingresos.


### 2.2.2 Estructura de los eventos de navegación

También se examinó la estructura de la tabla `clickstream`, que registra eventos generados durante la interacción de los usuarios con la plataforma.

A diferencia de las tablas transaccionales, esta entidad incorpora información semiestructurada mediante atributos anidados. Esta característica permite analizar simultáneamente el tipo de evento, el dispositivo utilizado y el origen de la interacción, además de relacionar la actividad con usuarios y propiedades.

La presencia de estructuras anidadas constituye un elemento relevante para la selección de la arquitectura, ya que requiere un modelo capaz de manejar información que no se limita a una estructura completamente plana.


In [0]:
%sql

DESCRIBE samples.wanderbricks.clickstream;

col_name,data_type,comment
event,string,The type of event that occurred: - 'view' (user views a property) - 'click' (user clicks on property or related element) - 'search' (user conducts a search) - 'filter' (user applies a filter to refine search results)
metadata,struct,"A nested dictionary containing additional details about the event: - 'device': The type of device used, e.g. 'desktop', 'mobile', 'tablet' - 'referrer': The source of the traffic, e.g 'google', 'direct', 'email', 'ad'"
property_id,bigint,The property that the user is interacting with
timestamp,timestamp,The timestamp of the interaction
user_id,bigint,The user that created/updated the booking


**Interpretación:** La tabla `clickstream` registra eventos de interacción de los usuarios con la plataforma, como visualizaciones, clics, búsquedas y aplicación de filtros. Además de los atributos convencionales, contiene el campo `metadata`, definido como una estructura `struct` que agrupa los atributos `device` y `referrer`.

Esta estructura permite conservar información relacionada con cada evento sin necesidad de convertir todos sus atributos en columnas independientes desde el origen. `property_id` y `user_id` permiten relacionar la actividad de navegación con propiedades y usuarios, mientras que `timestamp` incorpora la dimensión temporal del evento.

La presencia de este campo `struct` confirma que Wanderbricks combina datos estructurados con información semiestructurada, característica que será considerada posteriormente en la selección y justificación de la arquitectura Lakehouse.


### 2.3 Calidad de los datos

Después de revisar el volumen y la estructura de las principales entidades, se evaluó la calidad de los datos mediante comprobaciones de completitud. En primer lugar, se verificó la presencia de valores nulos en los atributos principales de la tabla `bookings`.

La comprobación compara el número total de registros con la cantidad de valores no nulos de cada campo. Esto permite identificar posibles atributos incompletos que podrían afectar las relaciones entre entidades o los análisis posteriores.


In [0]:
%sql

SELECT
    COUNT(*) AS total_registros,
    COUNT(booking_id) AS booking_id_no_nulos,
    COUNT(user_id) AS user_id_no_nulos,
    COUNT(property_id) AS property_id_no_nulos,
    COUNT(check_in) AS check_in_no_nulos,
    COUNT(check_out) AS check_out_no_nulos,
    COUNT(total_amount) AS total_amount_no_nulos,
    COUNT(status) AS status_no_nulos,
    COUNT(created_at) AS created_at_no_nulos
FROM samples.wanderbricks.bookings;

total_registros,booking_id_no_nulos,user_id_no_nulos,property_id_no_nulos,check_in_no_nulos,check_out_no_nulos,total_amount_no_nulos,status_no_nulos,created_at_no_nulos
72247,72247,72247,72247,72247,72247,72247,72247,72247


**Interpretación:** La tabla `bookings` contiene 72.247 registros y todos los campos evaluados presentan 72.247 valores no nulos. Por lo tanto, no se identificaron valores faltantes en los atributos principales analizados: identificadores, fechas, monto total, estado y fecha de creación.

Este resultado indica un alto nivel de completitud en la información de las reservas y reduce el riesgo de errores derivados de valores ausentes durante los análisis posteriores.


### 2.3.1 Verificación de registros duplicados

Además de comprobar la existencia de valores nulos, se verificó la unicidad del identificador `booking_id` en la tabla `bookings`.

La consulta agrupa los registros por `booking_id` y utiliza `HAVING COUNT(*) > 1` para identificar aquellos identificadores que aparecen más de una vez. Esta comprobación permite determinar si existen registros duplicados que puedan afectar el conteo de reservas y los análisis de ingresos.


In [0]:
%sql

SELECT
    booking_id,
    COUNT(*) AS cantidad
FROM samples.wanderbricks.bookings
GROUP BY booking_id
HAVING COUNT(*) > 1
ORDER BY cantidad DESC;

booking_id,cantidad


**Interpretación:** La consulta no devolvió registros, por lo que no se encontraron valores repetidos en `booking_id`. Esto confirma que, dentro de la tabla `bookings`, el identificador utilizado para representar cada reserva mantiene su unicidad.

La ausencia de duplicados facilita la realización de conteos, agregaciones de ingresos y relaciones con otras entidades sin introducir duplicaciones derivadas de reservas repetidas.


### 2.3.2 Verificación de consistencia temporal

Se realizó una comprobación de consistencia sobre las fechas de las reservas. En una reserva válida, la fecha de salida (`check_out`) debe ser posterior a la fecha de entrada (`check_in`).

Por esta razón, se identificaron los registros que cumplen la condición `check_out <= check_in`. Un resultado igual a cero indicaría que no se encontraron reservas con una secuencia temporal inválida.


In [0]:
%sql

SELECT
    COUNT(*) AS reservas_fecha_invalida
FROM samples.wanderbricks.bookings
WHERE check_out <= check_in;

reservas_fecha_invalida
0


**Interpretación:** El resultado fue `0`, por lo que no se encontraron reservas con una secuencia temporal inválida. En los 72.247 registros analizados, `check_out` es posterior a `check_in`.

Esta comprobación confirma la consistencia básica de las fechas de las reservas y reduce el riesgo de errores en análisis relacionados con duración de las estadías, comportamiento de los clientes e ingresos por reserva.


### 2.3.3 Verificación de valores monetarios

Se verificó la consistencia del atributo `total_amount`, que representa el valor total asociado a cada reserva.

La comprobación identifica los registros cuyo monto es menor o igual a cero. En el contexto de Wanderbricks, una reserva con un valor total no positivo podría representar un dato inconsistente y afectar los análisis de ingresos y rentabilidad.

Un resultado igual a cero indicaría que no se encontraron reservas con montos no válidos.


In [0]:
%sql

SELECT
    COUNT(*) AS montos_no_validos
FROM samples.wanderbricks.bookings
WHERE total_amount <= 0;

montos_no_validos
1


**Interpretación:** El resultado fue `1`, por lo que se identificó un registro cuyo `total_amount` es menor o igual a cero. Este caso representa una posible inconsistencia en los datos monetarios y debe considerarse durante la preparación de la capa Silver.

Aunque se trata de un único registro frente a los 72.247 registros de `bookings`, el hallazgo demuestra la importancia de realizar validaciones de calidad antes de utilizar los datos para análisis de ingresos. El registro deberá ser revisado y, según las reglas de negocio definidas, tratado durante el proceso de transformación.


### 2.3.4 Distribución de los valores monetarios

Como complemento a la validación de montos no positivos, se calcularon los valores mínimo, máximo y promedio de `total_amount`. Estas medidas permiten obtener una referencia general del comportamiento de los importes de las reservas y detectar posibles valores extremos.

Esta revisión es relevante porque los montos constituyen una de las principales variables para los análisis de ingresos que se realizarán posteriormente.


In [0]:
%sql

SELECT
    MIN(total_amount) AS monto_minimo,
    MAX(total_amount) AS monto_maximo,
    AVG(total_amount) AS monto_promedio
FROM samples.wanderbricks.bookings;

monto_minimo,monto_maximo,monto_promedio
-42.76,1178.46,553.7667607746087


**Interpretación:** El monto mínimo registrado es de aproximadamente `-42,76`, mientras que el máximo es de `1.178,46` y el promedio es de aproximadamente `553,77`.

El valor mínimo negativo confirma el hallazgo identificado en la validación anterior: existe al menos una reserva con un monto no válido según la regla de negocio establecida. Este caso deberá ser considerado durante la transformación de los datos hacia la capa Silver para evitar que un valor inconsistente afecte los análisis financieros.

El rango observado también permite establecer una referencia inicial sobre los valores monetarios de las reservas y facilita la identificación de posibles valores extremos durante las etapas posteriores del análisis.


### 2.3.5 Identificación del registro con monto no válido

A partir de la detección de un monto menor o igual a cero, se realizó una consulta específica para identificar el registro afectado y revisar sus principales atributos.

Esta revisión permite conservar evidencia del problema encontrado y determinar posteriormente cómo debe tratarse durante la transformación de los datos hacia la capa Silver.


In [0]:
%sql

SELECT
    booking_id,
    user_id,
    property_id,
    check_in,
    check_out,
    guests_count,
    total_amount,
    status,
    created_at
FROM samples.wanderbricks.bookings
WHERE total_amount <= 0;

booking_id,user_id,property_id,check_in,check_out,guests_count,total_amount,status,created_at
28499,77537,277,2025-06-13,2025-06-19,1,-42.76,pending,2025-05-19T14:50:26.000Z


**Interpretación:** Se identificó el registro `booking_id = 28499` como el único caso con un monto no válido. La reserva presenta un `total_amount` de aproximadamente `-42,76`, tiene estado `pending` y corresponde a una reserva de un huésped para una estadía entre el 13 y el 19 de junio de 2025.

El análisis permite confirmar que el valor negativo no corresponde a un comportamiento general de la tabla, sino a un caso puntual. No se establece una causa específica a partir de esta exploración, por lo que el registro será considerado durante la transformación y depuración de la capa Silver de acuerdo con las reglas de calidad definidas.


### 2.3.6 Distribución de estados de las reservas

Se analizó la distribución de los valores de `status` en la tabla `bookings` para identificar los estados presentes y conocer la cantidad de reservas asociadas a cada uno.

Esta revisión permite comprender el comportamiento de las reservas y establecer una referencia para los análisis posteriores. También permite contextualizar casos particulares encontrados durante la validación de calidad, como la reserva con monto negativo identificada anteriormente.


In [0]:
%sql

SELECT
    status,
    COUNT(*) AS cantidad
FROM samples.wanderbricks.bookings
GROUP BY status
ORDER BY cantidad DESC;

status,cantidad
pending,31575
confirmed,17948
cancelled,15285
completed,7439


**Interpretación:** La tabla `bookings` presenta cuatro estados de reserva: `pending`, `confirmed`, `cancelled` y `completed`. El estado con mayor cantidad de registros es `pending`, con 31.575 reservas, seguido de `confirmed` con 17.948, `cancelled` con 15.285 y `completed` con 7.439.

La suma de las cantidades corresponde a los 72.247 registros de la tabla, por lo que todos los registros analizados se encuentran asociados a uno de los estados identificados. Esta distribución proporciona contexto para los análisis posteriores sobre reservas, cancelaciones, ingresos y comportamiento del negocio.


### 2.4 Integridad de las relaciones entre tablas

Se verificó la integridad referencial entre la tabla `bookings` y la tabla `users`. Para ello, se realizó un `LEFT JOIN` utilizando `user_id` como campo de relación y se identificaron las reservas cuyo usuario asociado no existe en la tabla de usuarios.

Esta comprobación permite determinar si las reservas contienen referencias a usuarios inexistentes, situación que podría afectar la confiabilidad de los análisis que relacionan el comportamiento de los usuarios con sus reservas.


In [0]:
%sql

SELECT COUNT(*) AS reservas_sin_usuario
FROM samples.wanderbricks.bookings b
LEFT JOIN samples.wanderbricks.users u
    ON b.user_id = u.user_id
WHERE u.user_id IS NULL;

reservas_sin_usuario
0


**Interpretación:** El resultado fue `0`, por lo que no se encontraron reservas que hicieran referencia a usuarios inexistentes. Esto indica que los valores de `user_id` presentes en `bookings` tienen correspondencia en la tabla `users`.

La integridad de esta relación permite vincular de forma confiable las reservas con la información de los usuarios para realizar análisis posteriores sobre comportamiento, frecuencia de reservas e ingresos.


### 2.4.1 Relación entre reservas y propiedades

Se verificó la integridad de la relación entre `bookings` y `properties` mediante el campo `property_id`.

La consulta utiliza un `LEFT JOIN` para conservar todas las reservas y posteriormente identifica aquellas que no encuentran una propiedad correspondiente. Esta validación permite comprobar que las reservas pueden asociarse correctamente con las propiedades sobre las cuales se realizaron.


In [0]:
%sql

SELECT COUNT(*) AS reservas_sin_propiedad
FROM samples.wanderbricks.bookings b
LEFT JOIN samples.wanderbricks.properties p
    ON b.property_id = p.property_id
WHERE p.property_id IS NULL;

reservas_sin_propiedad
0


**Interpretación:** El resultado fue `0`, por lo que no se encontraron reservas que hicieran referencia a propiedades inexistentes. Todos los valores de `property_id` presentes en `bookings` tienen correspondencia en la tabla `properties`.

Esta integridad permite relacionar de manera confiable las reservas con las propiedades y facilita análisis posteriores sobre destinos, demanda, ingresos y desempeño de los alojamientos.


### 2.5 Calidad de datos en la tabla de pagos

Se realizó una validación de completitud sobre la tabla `payments`, revisando la presencia de valores no nulos en los principales atributos de cada pago.

Se evaluaron el identificador del pago, la reserva asociada, el monto, el método de pago, el estado y la fecha de pago. Esta comprobación permite identificar posibles datos faltantes que podrían afectar los análisis financieros y la relación entre pagos y reservas.


In [0]:
%sql

SELECT
    COUNT(*) AS total_registros,
    COUNT(payment_id) AS payment_id_no_nulos,
    COUNT(booking_id) AS booking_id_no_nulos,
    COUNT(amount) AS amount_no_nulos,
    COUNT(payment_method) AS payment_method_no_nulos,
    COUNT(status) AS status_no_nulos,
    COUNT(payment_date) AS payment_date_no_nulos
FROM samples.wanderbricks.payments;

total_registros,payment_id_no_nulos,booking_id_no_nulos,amount_no_nulos,payment_method_no_nulos,status_no_nulos,payment_date_no_nulos
49638,49638,49638,49638,49638,49638,49638


**Interpretación:** La tabla `payments` contiene 49.638 registros y cada uno de los atributos evaluados presenta 49.638 valores no nulos. Por lo tanto, no se identificaron valores faltantes en `payment_id`, `booking_id`, `amount`, `payment_method`, `status` ni `payment_date`.

La completitud de estos campos es importante para garantizar que las transacciones puedan relacionarse con sus reservas y que los análisis de ingresos y métodos de pago se realicen sobre información disponible.


### 2.5.1 Unicidad e integridad referencial de los pagos

Se verificó la calidad de los identificadores de la tabla `payments` y su relación con las reservas.

En primer lugar, se identificaron posibles valores duplicados en `payment_id`, que debe representar de forma única cada transacción. En segundo lugar, se comprobó que cada `booking_id` utilizado en `payments` tenga una reserva correspondiente en la tabla `bookings`.

Estas validaciones permiten determinar si los pagos pueden relacionarse de manera confiable con las reservas y si existen registros duplicados que puedan alterar los análisis financieros.


In [0]:
%sql

SELECT
    (
        SELECT COUNT(*)
        FROM (
            SELECT payment_id
            FROM samples.wanderbricks.payments
            GROUP BY payment_id
            HAVING COUNT(*) > 1
        )
    ) AS payment_ids_duplicados,

    (
        SELECT COUNT(*)
        FROM samples.wanderbricks.payments p
        LEFT JOIN samples.wanderbricks.bookings b
            ON p.booking_id = b.booking_id
        WHERE b.booking_id IS NULL
    ) AS pagos_sin_reserva;

payment_ids_duplicados,pagos_sin_reserva
4107,8


**Interpretación:** La consulta identificó dos situaciones que requieren revisión. Se encontraron **4.107 identificadores `payment_id` que aparecen más de una vez**, lo que evidencia una posible duplicación de registros en la tabla de pagos. Adicionalmente, se identificaron **8 pagos cuyo `booking_id` no tiene correspondencia en la tabla `bookings`**, lo que representa una inconsistencia en la relación entre pagos y reservas.

Estos hallazgos deben considerarse durante el proceso de transformación hacia la capa Silver, ya que los duplicados podrían sobreestimar el número de transacciones y los ingresos, mientras que los pagos sin una reserva asociada pueden afectar la integridad de los análisis financieros.


### 2.5.2 Cuantificación de los registros duplicados

Debido a que se identificaron 4.107 identificadores `payment_id` repetidos, se realizó una segunda comprobación para determinar cuántos registros de la tabla `payments` están involucrados en dichas duplicaciones.

Esta distinción permite evaluar con mayor precisión el impacto de la duplicidad sobre la calidad de los datos y evitar interpretar el número de identificadores duplicados como si correspondiera directamente al número de registros duplicados.


In [0]:
%sql

SELECT
    COUNT(*) AS registros_con_payment_id_duplicado
FROM samples.wanderbricks.payments
WHERE payment_id IN (
    SELECT payment_id
    FROM samples.wanderbricks.payments
    GROUP BY payment_id
    HAVING COUNT(*) > 1
);

registros_con_payment_id_duplicado
8572


**Interpretación:** El resultado fue de **8.572 registros involucrados en duplicaciones de `payment_id`**. Esto confirma que la situación identificada anteriormente no corresponde únicamente a unos pocos registros aislados: existen 4.107 identificadores de pago que aparecen más de una vez y que, en conjunto, involucran 8.572 registros.

Esta duplicidad puede generar sobreestimación de transacciones e ingresos si los registros se utilizan directamente en las consultas analíticas. Por esta razón, la depuración y definición del tratamiento de duplicados constituye un aspecto importante durante la construcción de la capa Silver.


### 2.5.3 Caracterización de los identificadores duplicados

Para profundizar en el hallazgo de duplicidad, se analizaron los `payment_id` que aparecen más de una vez. La consulta determina cuántas veces se repite cada identificador y calcula la primera y la última fecha registrada para cada uno.

Esta información permite caracterizar el comportamiento de los duplicados y establecer si las repeticiones corresponden a registros concentrados en una misma fecha o a eventos registrados en diferentes momentos.


In [0]:
%sql

SELECT
    payment_id,
    COUNT(*) AS veces,
    MIN(payment_date) AS primera_fecha,
    MAX(payment_date) AS ultima_fecha
FROM samples.wanderbricks.payments
GROUP BY payment_id
HAVING COUNT(*) > 1
ORDER BY veces DESC
LIMIT 10;

payment_id,veces,primera_fecha,ultima_fecha
17570283845213,4,2024-07-25T23:59:54.000Z,2025-05-27T10:52:05.000Z
17570283839240,4,2025-03-19T15:49:52.000Z,2025-07-29T10:44:05.000Z
17570283830924,4,2025-02-11T00:58:27.000Z,2025-07-18T17:48:15.000Z
17570283846716,4,2024-10-28T07:40:58.000Z,2025-07-19T08:49:57.000Z
17570283834124,4,2025-01-23T02:30:15.000Z,2025-07-26T05:09:17.000Z
17570283832417,4,2024-03-20T21:03:46.000Z,2025-06-19T23:46:24.000Z
17570283828299,4,2025-05-01T12:41:27.000Z,2025-07-23T23:59:59.000Z
17570283829000,4,2025-03-14T09:48:41.000Z,2025-07-25T18:18:36.000Z
17570283845612,4,2025-01-04T04:24:53.000Z,2025-07-30T00:36:47.000Z
17570283852795,4,2024-08-04T10:54:40.000Z,2025-07-16T15:18:49.000Z


**Interpretación:** Los identificadores analizados presentan hasta 4 apariciones y, en los casos observados, las fechas asociadas no son iguales. Por ejemplo, el `payment_id` `17570283845213` aparece cuatro veces entre julio de 2024 y mayo de 2025.

Por lo tanto, no se puede concluir únicamente con esta consulta que se trate de registros duplicados idénticos. El hallazgo indica una repetición de identificadores que requiere revisar los demás atributos de los registros antes de definir el tratamiento correspondiente en la capa Silver.

Esta situación evidencia la importancia de realizar validaciones de calidad sobre los identificadores y no asumir que un campo con nombre de identificador único necesariamente presenta unicidad en los datos de origen.


### 2.5.4 Revisión detallada de un identificador repetido

Se seleccionó uno de los identificadores repetidos para revisar directamente los registros asociados a él. Se eligió `payment_id = 17570283845213`, identificado previamente entre los casos con cuatro apariciones.

La consulta permite comparar los atributos de los registros, como `booking_id`, `amount`, `payment_method`, `status` y `payment_date`, y determinar si las apariciones corresponden a registros idénticos o contienen información diferente.


In [0]:
%sql

SELECT *
FROM samples.wanderbricks.payments
WHERE payment_id = 17570283845213
ORDER BY payment_date;

payment_id,booking_id,amount,payment_method,status,payment_date
17570283845213,35166,812.8800,apple_pay,completed,2024-07-25T23:59:54.000Z
17570283845213,23041,112.9500,apple_pay,completed,2025-02-04T19:14:35.000Z
17570283845213,31511,549.8900,bank_transfer,completed,2025-05-26T12:49:44.000Z
17570283845213,28276,963.6200,google_pay,completed,2025-05-27T10:52:05.000Z


**Interpretación:** La revisión detallada demuestra que las cuatro apariciones del `payment_id = 17570283845213` no corresponden a registros idénticos. Aunque comparten el mismo identificador, presentan diferentes `booking_id`, montos, métodos de pago y fechas.

Por lo tanto, el hallazgo no debe tratarse como una simple duplicación de filas para eliminar. Se interpreta como una inconsistencia en la unicidad del identificador `payment_id` dentro de los datos de origen. Esta situación debe quedar documentada y ser considerada al diseñar las reglas de transformación de la capa Silver, evitando eliminar registros válidos únicamente por presentar un identificador repetido.


2.5.5 Identificación de pagos sin reserva asociada
Debido a que se identificaron 8 registros de payments cuyo booking_id no tiene correspondencia en bookings, se realizó una consulta para identificar dichos registros y revisar sus principales atributos.

La consulta permite analizar el identificador del pago, la reserva referenciada, el monto, el método de pago, el estado y la fecha de la transacción. Esto permite caracterizar la inconsistencia y determinar su posible impacto sobre los análisis financieros.

In [0]:
%sql

SELECT
    p.payment_id,
    p.booking_id,
    p.amount,
    p.payment_method,
    p.status,
    p.payment_date
FROM samples.wanderbricks.payments p
LEFT JOIN samples.wanderbricks.bookings b
    ON p.booking_id = b.booking_id
WHERE b.booking_id IS NULL
ORDER BY p.payment_date;

payment_id,booking_id,amount,payment_method,status,payment_date
17570290012307,72653,965.6000,paypal,completed,2024-08-31T17:54:11.000Z
17570289986176,72663,668.5600,bank_transfer,completed,2025-02-01T19:47:49.000Z
17570289946976,72657,433.2400,credit_card,completed,2025-05-11T19:49:35.000Z
17570290111615,72672,377.8300,bank_transfer,completed,2025-06-14T19:12:46.000Z
17570289986173,72659,350.5800,apple_pay,completed,2025-06-27T10:56:42.000Z
17570290042795,72667,494.0100,credit_card,completed,2025-07-22T09:31:28.000Z
17570289999389,72664,918.5400,credit_card,completed,2025-07-30T18:48:11.000Z
17570289881126,72655,603.8000,credit_card,completed,2025-07-30T22:37:17.000Z


**Interpretación:** Se identificaron 8 pagos cuyo `booking_id` no tiene correspondencia en la tabla `bookings`. Los ocho registros presentan estado `completed` y montos positivos, por lo que la inconsistencia se encuentra específicamente en la relación entre el pago y la reserva referenciada.

Este hallazgo representa una inconsistencia de integridad referencial en los datos de origen. Los registros deben ser considerados durante la preparación de la capa Silver, ya que asociar estos pagos con reservas inexistentes podría afectar los análisis de ingresos y transacciones.


### 2.5.6 Rango de identificadores de las reservas

Como complemento a las validaciones de integridad referencial, se calculó el valor mínimo y máximo de `booking_id`, junto con el total de registros de la tabla `bookings`.

Esta información permite conocer el rango de los identificadores presentes y comprobar el volumen de reservas utilizado como referencia para relacionar los pagos con las reservas.


In [0]:
%sql

SELECT
    MIN(booking_id) AS booking_id_minimo,
    MAX(booking_id) AS booking_id_maximo,
    COUNT(*) AS total_reservas
FROM samples.wanderbricks.bookings;

booking_id_minimo,booking_id_maximo,total_reservas
1,72652,72247


**Interpretación:** La tabla `bookings` contiene 72.247 registros, con identificadores `booking_id` comprendidos entre 1 y 72.652.

Al comparar este rango con los 8 pagos identificados anteriormente sin reserva asociada, se observa que sus `booking_id` se encuentran por encima del máximo registrado en `bookings`. Esto confirma que dichos pagos hacen referencia a identificadores de reserva que actualmente no existen en la tabla de origen.

Este comportamiento constituye una inconsistencia de integridad referencial que deberá ser considerada durante la transformación de los datos hacia la capa Silver.


### 2.6 Análisis de datos semiestructurados en clickstream

La tabla `clickstream` contiene información semiestructurada mediante el campo `metadata`, definido como un `struct` que contiene los atributos `device` y `referrer`.

Se realizó una agregación combinando el tipo de evento con el dispositivo utilizado y la fuente de procedencia del tráfico. Esta consulta permite analizar cómo interactúan los usuarios con la plataforma según el dispositivo y el canal de adquisición, demostrando el manejo de estructuras anidadas dentro del modelo analítico.

Este tipo de estructura constituye una de las razones por las cuales el enfoque Lakehouse resulta adecuado para el caso de estudio, ya que permite trabajar con datos estructurados y semiestructurados dentro del mismo entorno.


In [0]:
%sql

SELECT
    event,
    metadata.device AS device,
    metadata.referrer AS referrer,
    COUNT(*) AS cantidad
FROM samples.wanderbricks.clickstream
GROUP BY
    event,
    metadata.device,
    metadata.referrer
ORDER BY cantidad DESC;

event,device,referrer,cantidad
click,desktop,google,2193
search,tablet,direct,2180
view,desktop,email,2155
search,tablet,ad,2147
filter,tablet,direct,2136
click,tablet,email,2135
search,desktop,email,2134
click,tablet,ad,2133
view,tablet,email,2133
filter,tablet,email,2131


**Interpretación:** La consulta demuestra que es posible acceder directamente a los atributos anidados de `metadata` mediante `metadata.device` y `metadata.referrer`. Entre las combinaciones observadas se encuentran eventos `click`, `search`, `view` y `filter`, realizados desde dispositivos desktop, tablet y mobile, y provenientes de fuentes como `google`, `direct`, `email` y `ad`.

La combinación con mayor frecuencia dentro de la salida observada corresponde a `click` desde un dispositivo desktop y con procedencia `google`, con 2.193 eventos. Esta variedad evidencia que el dataset contiene información de navegación estructurada junto con atributos semiestructurados, lo cual debe considerarse en el diseño de la base de datos analítica.


### 2.6.1 Completitud de los datos semiestructurados

Se realizó una validación de completitud sobre la tabla `clickstream`, incluyendo tanto la columna estructurada `metadata` como sus atributos internos `device` y `referrer`.

También se verificó la presencia de valores en los campos `event`, `property_id`, `timestamp` y `user_id`. Esta comprobación permite determinar si los eventos de navegación cuentan con la información necesaria para realizar análisis de comportamiento, dispositivos y fuentes de tráfico.


In [0]:
%sql

SELECT
    COUNT(*) AS total_registros,
    COUNT(metadata) AS metadata_no_nulo,
    COUNT(metadata.device) AS device_no_nulo,
    COUNT(metadata.referrer) AS referrer_no_nulo,
    COUNT(event) AS event_no_nulo,
    COUNT(property_id) AS property_id_no_nulo,
    COUNT(timestamp) AS timestamp_no_nulo,
    COUNT(user_id) AS user_id_no_nulo
FROM samples.wanderbricks.clickstream;

total_registros,metadata_no_nulo,device_no_nulo,referrer_no_nulo,event_no_nulo,property_id_no_nulo,timestamp_no_nulo,user_id_no_nulo
100000,100000,100000,100000,100000,100000,100000,100000


**Interpretación:** La tabla `clickstream` contiene 100.000 registros y todos los campos evaluados presentan 100.000 valores no nulos. Esto incluye la estructura `metadata` y sus atributos internos `device` y `referrer`, además de `event`, `property_id`, `timestamp` y `user_id`.

Por lo tanto, no se identificaron valores faltantes en los atributos evaluados. La completitud de `metadata` es especialmente relevante porque permite utilizar de forma confiable la información semiestructurada para analizar los dispositivos y las fuentes de tráfico de los usuarios.


### 2.7 Estructura de la tabla de reseñas

Se examinó la estructura de la tabla `reviews` para identificar sus atributos, tipos de datos y relaciones con otras entidades del marketplace.

Las reseñas son relevantes para el análisis del negocio porque permiten complementar las variables transaccionales con información relacionada con la experiencia y valoración de los usuarios.


In [0]:
%sql

DESCRIBE samples.wanderbricks.reviews;

col_name,data_type,comment
booking_id,bigint,The booking that was reviewed
comment,string,User's review comment
created_at,timestamp,The date the review was created
is_deleted,boolean,Whether the review was deleted
property_id,bigint,The property that was reviewed
rating,double,The rating given by the user (1.0-5.0)
review_id,bigint,Uniquely identifies this review
updated_at,timestamp,The date the review was last updated
user_id,bigint,The user that wrote the review


**Interpretación:** La tabla `reviews` contiene información estructurada y texto libre. Sus principales atributos incluyen identificadores de la reseña, reserva, propiedad y usuario; una valoración numérica (`rating`); el comentario textual del usuario (`comment`); un indicador de eliminación lógica (`is_deleted`) y fechas de creación y actualización.

La presencia de la columna `comment` aporta una dimensión de texto libre al conjunto de datos, mientras que `rating` permite realizar análisis cuantitativos sobre la satisfacción de los usuarios. Además, los campos `booking_id`, `property_id` y `user_id` permiten relacionar las reseñas con otras entidades del marketplace.


### 2.8 Estructura de la tabla de propiedades

Se examinó la estructura de la tabla `properties` para identificar los atributos que describen los alojamientos disponibles en el marketplace y los campos utilizados para relacionarlos con otras entidades.

Esta tabla representa la oferta de alojamientos y resulta fundamental para analizar reservas, ingresos, características de las propiedades y desempeño de los anfitriones.


In [0]:
%sql

DESCRIBE samples.wanderbricks.properties;

col_name,data_type,comment
property_id,bigint,Unique identifier of this property
host_id,bigint,The identifier of the host of this property
destination_id,bigint,Foreign Key to 'destinations'
title,string,Title of the property listing
description,string,Description of the property
base_price,float,Base price for booking
property_type,string,"Type of property (e.g., house, apartment)"
max_guests,int,Maximum number of guests allowed
bedrooms,int,Number of bedrooms
bathrooms,int,Number of bathrooms


**Interpretación:** La tabla `properties` contiene 13 atributos que describen los alojamientos del marketplace. Incluye identificadores y relaciones con anfitriones y destinos mediante `host_id` y `destination_id`, además de información descriptiva como `title`, `description` y `property_type`.

También contiene variables numéricas relacionadas con la oferta, como `base_price`, `max_guests`, `bedrooms` y `bathrooms`, así como coordenadas geográficas mediante `property_latitude` y `property_longitude`. Finalmente, `created_at` registra la fecha en que fue publicada cada propiedad.

La combinación de información descriptiva, numérica, relacional y geográfica demuestra la variedad de datos disponibles para construir análisis sobre la oferta de alojamientos y su relación con las reservas y los destinos.


2.9 Estructura de la tabla de destinos

Se examinó la estructura de la tabla destinations, que representa la dimensión geográfica del marketplace. Esta información permite asociar las propiedades con un destino determinado y posteriormente analizar variables como cantidad de reservas, ingresos y nivel de demanda por ubicación.

In [0]:
%sql
DESCRIBE samples.wanderbricks.destinations;

col_name,data_type,comment
destination_id,bigint,Primary Key
destination,string,Name of the destination
country,string,Name of the country
state_or_province,string,"The name of the state, province or region the destination is located in"
state_or_province_code,string,"The code of the state, province or region the destination is located in"
description,string,"Detailed description of the destination. This column has text data (Markdown format) containing summary information of the destination about tourist destinations, destination demographics, historical and geographical information, and popular cuisine you can find in the destination. Additionally, tourists might find transportation and travel guidelines helpful."


**Interpretación:** La tabla `destinations` contiene información geográfica y descriptiva de los destinos disponibles en el marketplace. Cuenta con un identificador principal (`destination_id`), el nombre del destino, país, estado o provincia y su respectivo código regional.

Además, el campo `description` contiene información textual detallada en formato Markdown sobre los destinos, incluyendo aspectos turísticos, demográficos, históricos, geográficos, gastronómicos y de transporte. Esto amplía la variedad del conjunto de datos al incorporar contenido descriptivo de texto junto con información geográfica estructurada.

La tabla puede relacionarse con `properties` mediante `destination_id`, permitiendo analizar las propiedades, reservas e ingresos según su ubicación geográfica.


### 2.10 Estructura de la tabla de usuarios

Se examinó la estructura de la tabla `users` para identificar los atributos que caracterizan a los usuarios del marketplace y los campos que permiten relacionarlos con sus actividades dentro de la plataforma.

Esta información es relevante para analizar el comportamiento de los usuarios y establecer relaciones con las reservas, reseñas y eventos de navegación registrados en otras tablas del conjunto de datos.


In [0]:
%sql

DESCRIBE samples.wanderbricks.users;

col_name,data_type,comment
user_id,bigint,Unique identifier of the user
email,string,User's email address.
name,string,Name of the user.
country,string,Country of residence.
user_type,string,"Type of user (e.g., individual or business)."
created_at,timestamp,Account creation date.
is_business,boolean,Boolean indicating if the user is a business.
company_name,string,Name of the company (if user is a business).


**Interpretación:** La tabla `users` contiene información estructurada sobre los usuarios del marketplace. El campo `user_id` identifica de forma única a cada usuario y permite relacionarlo con otras entidades, mientras que `email`, `name` y `country` contienen información descriptiva.

También incluye atributos de clasificación como `user_type` e `is_business`, además de `company_name` para los usuarios empresariales. El campo `created_at` permite analizar la incorporación de usuarios a lo largo del tiempo.

La estructura de esta tabla permite complementar los análisis de reservas y comportamiento de navegación con características de los usuarios, manteniendo relaciones mediante el identificador `user_id`.


### 2.11 Estructura de la tabla de anfitriones

Se examinó la estructura de la tabla `hosts` para identificar los atributos de los anfitriones y los campos que permiten relacionarlos con las propiedades ofrecidas en el marketplace.

Esta información permite complementar el análisis de la oferta de alojamientos con características y métricas de desempeño de los anfitriones.


In [0]:
%sql

DESCRIBE samples.wanderbricks.hosts;

col_name,data_type,comment
host_id,bigint,Unique identifier of the host
name,string,Host's full name
email,string,Hosts's email address
phone,string,Host's phone number
is_verified,boolean,Verification status of the host
is_active,boolean,Active status of the host
rating,float,Host rating (1.0-5.0)
country,string,Host's country of residence
joined_at,date,Date the host joined


**Interpretación:** La tabla `hosts` contiene información estructurada sobre los anfitriones del marketplace. El campo `host_id` identifica de manera única a cada anfitrión y permite relacionarlo con las propiedades que administra.

La tabla incluye información descriptiva como nombre, correo electrónico y teléfono, además de atributos de estado como `is_verified` e `is_active`. También incorpora `rating`, que permite analizar la valoración de los anfitriones, `country` para caracterizar su ubicación y `joined_at` para estudiar su incorporación a la plataforma.

La relación entre `hosts` y `properties` permite analizar el desempeño de los anfitriones a partir de variables como número de propiedades, reservas e ingresos generados.


### 2.11.1 Validación de la relación entre propiedades y anfitriones

Se verificó la integridad de la relación entre las tablas `properties` y `hosts`, utilizando `host_id` como campo de correspondencia.

La consulta identifica propiedades cuyo anfitrión referenciado no existe en la tabla `hosts`. Esta validación permite determinar si la oferta de alojamientos cuenta con un anfitrión válido asociado y detectar posibles inconsistencias en las relaciones del conjunto de datos.


In [0]:
%sql

SELECT COUNT(*) AS propiedades_sin_host
FROM samples.wanderbricks.properties p
LEFT JOIN samples.wanderbricks.hosts h
    ON p.host_id = h.host_id
WHERE h.host_id IS NULL;

propiedades_sin_host
0


**Interpretación:** El resultado fue de **0 propiedades sin anfitrión asociado**. Esto indica que todos los `host_id` presentes en `properties` tienen correspondencia en la tabla `hosts`.

Por lo tanto, no se identificaron inconsistencias de integridad referencial entre estas dos tablas en la validación realizada.


### 2.11.2 Validación de la relación entre propiedades y destinos

Se verificó la integridad referencial entre `properties` y `destinations` mediante el campo `destination_id`.

La consulta identifica propiedades cuyo destino asociado no existe en la tabla de destinos. Esta validación permite comprobar que las propiedades cuentan con una ubicación geográfica válida dentro de la información disponible.


In [0]:
%sql

SELECT COUNT(*) AS propiedades_sin_destino
FROM samples.wanderbricks.properties p
LEFT JOIN samples.wanderbricks.destinations d
    ON p.destination_id = d.destination_id
WHERE d.destination_id IS NULL;

propiedades_sin_destino
0


**Interpretación:** El resultado fue de **0 propiedades sin destino asociado**. Esto indica que todos los `destination_id` presentes en `properties` tienen correspondencia en la tabla `destinations`.

Por lo tanto, no se identificaron inconsistencias de integridad referencial entre propiedades y destinos en la validación realizada. Esta relación permite mantener la trazabilidad geográfica desde una propiedad hasta su destino.


### 2.12 Estructura de la tabla de países

Se examinó la estructura de la tabla `countries` para identificar los atributos disponibles relacionados con los países y complementar la dimensión geográfica del conjunto de datos.

Esta información permite ampliar los análisis de destinos y propiedades incorporando características asociadas al país correspondiente.


In [0]:
%sql

DESCRIBE samples.wanderbricks.countries;

col_name,data_type,comment
country,string,The name of the country (Primary Key
country_code,string,The ISO country code (2-letter)
continent,string,The continent the country is located in


**Interpretación:** La tabla `countries` contiene una dimensión geográfica de referencia compuesta por el nombre del país, su código ISO de dos letras y el continente al que pertenece. El campo `country` funciona como clave primaria.

Esta información permite complementar los datos de destinos y usuarios con una clasificación geográfica estandarizada por país y continente, facilitando análisis regionales y comparaciones entre mercados.


# 1. Exploración y descripción de los datos

El caso de estudio seleccionado es Wanderbricks, un marketplace de alquiler vacacional. El conjunto de datos integra información de usuarios, anfitriones, propiedades, reservas, pagos, reseñas, navegación y dimensiones geográficas.

La exploración inicial se realizó sobre las tablas disponibles en el catálogo `samples.wanderbricks`, con el propósito de conocer su volumen, estructura, variedad, calidad y relaciones antes de definir el modelo de datos analítico.

## 1.1 Volumen de los datos

Durante la exploración se identificaron los siguientes volúmenes:

| Tabla | Registros |
|---|---:|
| users | 124.509 |
| hosts | 19.384 |
| properties | 18.163 |
| bookings | 72.247 |
| payments | 49.638 |
| reviews | 99.793 |
| clickstream | 100.000 |
| page_views | 500.000 |
| destinations | 42 |
| countries | 168 |

Se observa una variedad importante de tamaños. Las tablas de eventos presentan los mayores volúmenes, especialmente `page_views` con 500.000 registros y `clickstream` con 100.000 registros, mientras que las dimensiones geográficas son considerablemente menores.

## 1.2 Variedad y tipos de datos

El dataset combina diferentes tipos de información:

- **Datos estructurados:** usuarios, anfitriones, propiedades, reservas y pagos.
- **Datos transaccionales:** `bookings` y `payments`.
- **Eventos de navegación:** `clickstream` y `page_views`.
- **Texto libre:** comentarios de `reviews`, descripciones de propiedades y descripciones de destinos.
- **Datos geográficos:** propiedades con latitud y longitud, destinos y países.
- **Datos semiestructurados:** `clickstream.metadata`, definido como un `STRUCT` con los atributos `device` y `referrer`.

Esta variedad requiere un modelo capaz de trabajar con estructuras tabulares tradicionales y, al mismo tiempo, con información semiestructurada y textual.

## 1.3 Relaciones entre las entidades

Las principales relaciones identificadas son:

- `bookings.user_id` → `users.user_id`
- `bookings.property_id` → `properties.property_id`
- `properties.host_id` → `hosts.host_id`
- `properties.destination_id` → `destinations.destination_id`
- `payments.booking_id` → `bookings.booking_id`
- `reviews.booking_id` → `bookings.booking_id`
- `reviews.property_id` → `properties.property_id`
- `reviews.user_id` → `users.user_id`
- `clickstream.user_id` → `users.user_id`
- `clickstream.property_id` → `properties.property_id`

Estas relaciones muestran que el modelo contiene entidades relacionadas mediante identificadores, pero también incorpora información de eventos y estructuras anidadas.

## 1.4 Calidad observada

Durante la exploración se realizaron diferentes validaciones de calidad.

En `bookings` se analizaron 72.247 registros. No se encontraron valores nulos en los campos principales evaluados, tampoco se encontraron `booking_id` duplicados ni inconsistencias donde `check_out` fuera anterior o igual a `check_in`. Además, no se encontraron reservas sin usuario o sin propiedad asociada.

Sin embargo, se identificó una anomalía en `total_amount`: existe un registro con valor negativo de aproximadamente -42,76. El registro corresponde al `booking_id` 28499 y se encuentra en estado `pending`.

En `payments` no se encontraron valores nulos en los campos evaluados. No obstante, se identificaron 4.107 valores de `payment_id` que aparecen en múltiples registros, involucrando 8.572 filas. La inspección de un caso mostró que los registros repetidos no son idénticos, sino que corresponden a diferentes reservas, montos, métodos de pago y fechas. Esto evidencia una diferencia entre la unicidad indicada en la documentación y la granularidad observada en los datos.

También se identificaron 8 pagos cuyo `booking_id` no tiene correspondencia en `bookings`. Estos identificadores se encuentran por encima del máximo `booking_id` registrado en `bookings`, que es 72.652.

Estos hallazgos justifican la aplicación de reglas de validación y transformación durante la construcción de la capa Silver, manteniendo inicialmente los datos originales en Bronze.

## 1.5 Datos semiestructurados y texto libre

La tabla `clickstream` contiene 100.000 registros y presenta una estructura anidada mediante el campo `metadata`, definido como:

`STRUCT<device:STRING, referrer:STRING>`

Durante la exploración se comprobó que los atributos `metadata.device` y `metadata.referrer` pueden consultarse directamente. Se identificaron tres tipos de dispositivo (`desktop`, `tablet` y `mobile`) y cuatro fuentes de referencia (`google`, `direct`, `email` y `ad`).

Además, la tabla `reviews` contiene el campo `comment` de tipo `string`, utilizado para almacenar comentarios de los usuarios. También existen campos de texto descriptivo en `properties` y `destinations`.

Esto demuestra que el conjunto de datos combina estructuras tabulares con información semiestructurada y texto libre.

# 2. Justificación del modelo de datos

A partir de las características identificadas durante la exploración, se compararon tres alternativas de modelado: paradigma relacional, NoSQL y Lakehouse.

## 2.1 Paradigma relacional

El modelo relacional resulta adecuado para una parte importante de Wanderbricks debido a la existencia de relaciones claras entre usuarios, reservas, propiedades, anfitriones, pagos y destinos.

La integridad referencial es especialmente relevante en entidades como `bookings`, donde `user_id` y `property_id` permiten relacionar cada reserva con un usuario y una propiedad.

Sin embargo, el dataset también contiene estructuras que requieren mayor flexibilidad, como el campo `metadata` de `clickstream`, definido como un `STRUCT`, además de texto libre en reseñas y descripciones. Por esta razón, aunque el modelo relacional resulta adecuado para los datos estructurados y transaccionales, presenta una menor flexibilidad frente a la variedad completa del dataset.

## 2.2 Paradigma NoSQL

El paradigma NoSQL resulta atractivo para los datos semiestructurados de Wanderbricks. Por ejemplo, `clickstream.metadata` contiene información anidada sobre el dispositivo y la fuente de referencia, una estructura que puede representarse naturalmente mediante documentos.

Sin embargo, Wanderbricks posee numerosas relaciones entre entidades y requiere análisis que combinan reservas, pagos, usuarios, propiedades, reseñas y dimensiones geográficas. Estas relaciones y operaciones analíticas hacen que un enfoque exclusivamente NoSQL no sea la alternativa más conveniente para integrar todo el escenario analítico.

## 2.3 Lakehouse

Se selecciona el paradigma Lakehouse como modelo de implementación para Wanderbricks.

La decisión se fundamenta en las características observadas durante la exploración: el dataset combina datos estructurados, datos transaccionales, eventos de navegación, texto libre, información geográfica y estructuras semiestructuradas.

Además, existen diferentes volúmenes de información, desde dimensiones pequeñas como `destinations` con 42 registros hasta `page_views` con 500.000 registros. También se identificaron problemas de calidad que requieren conservar el dato original y aplicar procesos de transformación y validación.

El enfoque Lakehouse permite conservar los datos originales en una capa Bronze y construir posteriormente una capa Silver con reglas de calidad y estructuras adecuadas para el análisis. El uso de tablas Delta permite además trabajar con transacciones ACID, historial de versiones y evolución de esquema, características que hacen que este paradigma se ajuste a las necesidades del caso y a los requisitos de la actividad.

## 2.4 Decisión de diseño

La solución utilizará un modelo Lakehouse basado en tablas Delta, organizado inicialmente en capas Bronze y Silver.

La capa Bronze conservará los datos originales provenientes del conjunto de datos fuente. La capa Silver aplicará transformaciones, validaciones de calidad y normalización de los datos cuando sea necesario, manteniendo estructuras semiestructuradas cuando aporten valor al análisis.

La elección del Lakehouse permite combinar las ventajas del procesamiento analítico con la flexibilidad necesaria para trabajar con datos estructurados, semiestructurados y textuales.

## 3. Implementación de la base de datos analítica

### 3.1 Creación de la arquitectura del Lakehouse

A partir de las características identificadas durante la exploración, se implementó la arquitectura inicial del Lakehouse utilizando un catálogo propio para el proyecto.

Se creó el catálogo `wanderbricks_analitica` y se organizaron los datos mediante las capas `bronze` y `silver`. La capa Bronze incluye además el volumen `raw_data`, destinado a almacenar los datos crudos provenientes del conjunto de datos fuente.

Esta organización permite separar los datos originales de los procesos posteriores de transformación y limpieza, facilitando la trazabilidad y el procesamiento por etapas.


In [0]:
%sql

CREATE CATALOG IF NOT EXISTS wanderbricks_analitica;

In [0]:
%sql

CREATE SCHEMA IF NOT EXISTS wanderbricks_analitica.bronze;

In [0]:
%sql

CREATE VOLUME IF NOT EXISTS wanderbricks_analitica.bronze.raw_data;

In [0]:
%sql

CREATE SCHEMA IF NOT EXISTS wanderbricks_analitica.silver;

### 3.2 Creación de las tablas de la capa Bronze

Se construyeron las tablas de la capa Bronze utilizando el formato Delta y definiendo explícitamente el esquema de cada tabla.

La tabla `bookings` conserva la estructura de los datos originales de reservas, incluyendo identificadores, fechas, cantidad de huéspedes, monto, estado y marcas de tiempo. En esta primera capa no se aplican transformaciones ni reglas de limpieza, con el propósito de conservar los datos de origen y mantener su trazabilidad.

El uso de Delta permite disponer de las capacidades transaccionales y de versionado requeridas para la implementación del Lakehouse.


In [0]:
%sql

CREATE TABLE IF NOT EXISTS wanderbricks_analitica.bronze.bookings (
    booking_id BIGINT,
    user_id BIGINT,
    property_id BIGINT,
    check_in DATE,
    check_out DATE,
    guests_count INT,
    total_amount FLOAT,
    status STRING,
    created_at TIMESTAMP,
    updated_at TIMESTAMP
)
USING DELTA;

### 3.3 Ingestión de los datos en la capa Bronze

Una vez creadas las tablas Delta con esquema explícito, se realizó la ingestión de los datos desde el catálogo fuente `samples.wanderbricks` hacia las tablas correspondientes de la capa Bronze.

En esta etapa se conservan los registros provenientes de la fuente sin aplicar transformaciones de limpieza o corrección. Esto permite mantener una representación de los datos originales y disponer de una base de referencia para las transformaciones posteriores en la capa Silver.

Para `bookings`, la correspondencia entre las columnas de la fuente y la tabla Bronze se especifica directamente mediante la sentencia `INSERT INTO`.


In [0]:
%sql

INSERT INTO wanderbricks_analitica.bronze.bookings
SELECT
    booking_id,
    user_id,
    property_id,
    check_in,
    check_out,
    guests_count,
    total_amount,
    status,
    created_at,
    updated_at
FROM samples.wanderbricks.bookings;

num_affected_rows,num_inserted_rows
72247,72247


**Validación de la ingestión:** Se verificó el número de registros cargados en la tabla `bronze.bookings` para comprobar que la ingestión desde `samples.wanderbricks.bookings` se realizó correctamente.


In [0]:
%sql

SELECT COUNT(*) AS total_registros
FROM wanderbricks_analitica.bronze.bookings;

total_registros
216741


**Resultado:** La tabla `wanderbricks_analitica.bronze.bookings` contiene 72.247 registros, coincidiendo con el volumen identificado en la tabla fuente `samples.wanderbricks.bookings`. Esto confirma que la totalidad de los registros de reservas fue ingerida correctamente en la capa Bronze.


In [0]:
%sql

CREATE TABLE IF NOT EXISTS wanderbricks_analitica.bronze.payments (
    payment_id BIGINT,
    booking_id BIGINT,
    amount DECIMAL(15,4),
    payment_method STRING,
    status STRING,
    payment_date TIMESTAMP
)
USING DELTA;

In [0]:
%sql

INSERT INTO wanderbricks_analitica.bronze.payments
SELECT
    payment_id,
    booking_id,
    amount,
    payment_method,
    status,
    payment_date
FROM samples.wanderbricks.payments;

num_affected_rows,num_inserted_rows
49638,49638


In [0]:
%sql

SELECT COUNT(*) AS total_registros
FROM wanderbricks_analitica.bronze.payments;

total_registros
99276


**Resultado:** La tabla `wanderbricks_analitica.bronze.payments` contiene 49.638 registros, coincidiendo con el volumen de la tabla fuente `samples.wanderbricks.payments`. Esto confirma que la totalidad de los registros de pagos fue ingerida correctamente en la capa Bronze.


**Corrección de la carga:** Durante la validación de la ingestión se detectó que los registros de `payments` habían sido cargados dos veces, generando 99.276 registros en Bronze frente a los 49.638 existentes en la fuente.

Para corregir la duplicación se vació la tabla `bronze.payments` mediante `TRUNCATE TABLE`, conservando su estructura para realizar nuevamente la ingestión desde la fuente.


In [0]:
%sql

TRUNCATE TABLE wanderbricks_analitica.bronze.payments;

In [0]:
%sql

INSERT INTO wanderbricks_analitica.bronze.payments
SELECT
    payment_id,
    booking_id,
    amount,
    payment_method,
    status,
    payment_date
FROM samples.wanderbricks.payments;

num_affected_rows,num_inserted_rows
49638,49638


In [0]:
%sql

SELECT COUNT(*) AS total_registros
FROM wanderbricks_analitica.bronze.payments;

total_registros
49638


**Resultado:** Después de vaciar y recargar la tabla, `bronze.payments` contiene 49.638 registros, coincidiendo con la cantidad de registros de la fuente. Se confirma así que la duplicación detectada durante la primera carga fue corregida y que la tabla Bronze conserva nuevamente el volumen original de `payments`.


In [0]:
%sql

CREATE TABLE IF NOT EXISTS wanderbricks_analitica.bronze.reviews (
    booking_id BIGINT,
    comment STRING,
    created_at TIMESTAMP,
    is_deleted BOOLEAN,
    property_id BIGINT,
    rating DOUBLE,
    review_id BIGINT,
    updated_at TIMESTAMP,
    user_id BIGINT
)
USING DELTA;

In [0]:
%sql

INSERT INTO wanderbricks_analitica.bronze.reviews
SELECT
    booking_id,
    comment,
    created_at,
    is_deleted,
    property_id,
    rating,
    review_id,
    updated_at,
    user_id
FROM samples.wanderbricks.reviews;

num_affected_rows,num_inserted_rows
99793,99793


In [0]:
%sql

SELECT COUNT(*) AS total_registros
FROM wanderbricks_analitica.bronze.reviews;

total_registros
299379


**Resultado:** La tabla `wanderbricks_analitica.bronze.reviews` contiene 99.793 registros, coincidiendo con el volumen de la tabla fuente `samples.wanderbricks.reviews`. Esto confirma que los registros de reseñas fueron ingeridos correctamente en la capa Bronze, conservando también el campo de texto libre `comment`.


In [0]:
%sql

CREATE TABLE IF NOT EXISTS wanderbricks_analitica.bronze.clickstream (
    event STRING,
    metadata STRUCT<
        device: STRING,
        referrer: STRING
    >,
    property_id BIGINT,
    timestamp TIMESTAMP,
    user_id BIGINT
)
USING DELTA;

In [0]:
%sql

INSERT INTO wanderbricks_analitica.bronze.clickstream
SELECT
    event,
    metadata,
    property_id,
    timestamp,
    user_id
FROM samples.wanderbricks.clickstream;

num_affected_rows,num_inserted_rows
100000,100000


In [0]:
%sql

SELECT COUNT(*) AS total_registros
FROM wanderbricks_analitica.bronze.clickstream;

total_registros
300000


**Resultado:** La tabla `wanderbricks_analitica.bronze.clickstream` contiene 100.000 registros, coincidiendo con el volumen de la tabla fuente `samples.wanderbricks.clickstream`. La ingestión conserva además la estructura semiestructurada `metadata` como un `STRUCT` con los atributos `device` y `referrer`.


**Validación de la estructura semiestructurada:** Se verificó que la columna `metadata` conserva su estructura `STRUCT` después de la ingestión en la capa Bronze. Mediante la notación `metadata.device` y `metadata.referrer` se accede directamente a los atributos internos, comprobando que la información semiestructurada fue trasladada correctamente desde la fuente.


In [0]:
%sql

SELECT
    event,
    metadata.device AS dispositivo,
    metadata.referrer AS origen,
    property_id,
    user_id
FROM wanderbricks_analitica.bronze.clickstream
LIMIT 10;

event,dispositivo,origen,property_id,user_id
view,desktop,email,5135,16390
search,desktop,direct,7805,3655
click,mobile,direct,17349,100094
filter,tablet,ad,6128,102187
search,mobile,google,3438,40863
search,mobile,email,14475,115938
search,tablet,ad,4788,69029
view,desktop,direct,12099,37389
view,mobile,google,6600,534
click,mobile,google,13528,107117


In [0]:
%sql

CREATE TABLE IF NOT EXISTS wanderbricks_analitica.bronze.properties (
    property_id BIGINT,
    host_id BIGINT,
    destination_id BIGINT,
    title STRING,
    description STRING,
    base_price FLOAT,
    property_type STRING,
    max_guests INT,
    bedrooms INT,
    bathrooms INT,
    property_latitude FLOAT,
    property_longitude FLOAT,
    created_at DATE
)
USING DELTA;

In [0]:
%sql

INSERT INTO wanderbricks_analitica.bronze.properties
SELECT
    property_id,
    host_id,
    destination_id,
    title,
    description,
    base_price,
    property_type,
    max_guests,
    bedrooms,
    bathrooms,
    property_latitude,
    property_longitude,
    created_at
FROM samples.wanderbricks.properties;

num_affected_rows,num_inserted_rows
18163,18163


In [0]:
%sql

SELECT COUNT(*) AS total_registros
FROM wanderbricks_analitica.bronze.properties;

total_registros
54489


In [0]:
%sql

CREATE TABLE IF NOT EXISTS wanderbricks_analitica.bronze.destinations (
    destination_id BIGINT,
    destination STRING,
    country STRING,
    state_or_province STRING,
    state_or_province_code STRING,
    description STRING
)
USING DELTA;

In [0]:
%sql

INSERT INTO wanderbricks_analitica.bronze.destinations
SELECT
    destination_id,
    destination,
    country,
    state_or_province,
    state_or_province_code,
    description
FROM samples.wanderbricks.destinations;

num_affected_rows,num_inserted_rows
42,42


In [0]:
%sql

SELECT COUNT(*) AS total_registros
FROM wanderbricks_analitica.bronze.destinations;

total_registros
126


In [0]:
%sql

CREATE TABLE IF NOT EXISTS wanderbricks_analitica.bronze.countries (
    country STRING,
    country_code STRING,
    continent STRING
)
USING DELTA;

In [0]:
%sql

INSERT INTO wanderbricks_analitica.bronze.countries
SELECT
    country,
    country_code,
    continent
FROM samples.wanderbricks.countries;

num_affected_rows,num_inserted_rows
168,168


In [0]:
%sql

SELECT COUNT(*) AS total_registros
FROM wanderbricks_analitica.bronze.countries;

total_registros
504


In [0]:
%sql

CREATE TABLE IF NOT EXISTS wanderbricks_analitica.bronze.users (
    user_id BIGINT,
    email STRING,
    name STRING,
    country STRING,
    user_type STRING,
    created_at TIMESTAMP,
    is_business BOOLEAN,
    company_name STRING
)
USING DELTA;

In [0]:
%sql

INSERT INTO wanderbricks_analitica.bronze.users
SELECT
    user_id,
    email,
    name,
    country,
    user_type,
    created_at,
    is_business,
    company_name
FROM samples.wanderbricks.users;

num_affected_rows,num_inserted_rows
124509,124509


In [0]:
%sql

SELECT COUNT(*) AS total_registros
FROM wanderbricks_analitica.bronze.users;

total_registros
373527


In [0]:
%sql

CREATE TABLE IF NOT EXISTS wanderbricks_analitica.bronze.hosts (
    host_id BIGINT,
    name STRING,
    email STRING,
    phone STRING,
    is_verified BOOLEAN,
    is_active BOOLEAN,
    rating FLOAT,
    country STRING,
    joined_at DATE
)
USING DELTA;

In [0]:
%sql

INSERT INTO wanderbricks_analitica.bronze.hosts
SELECT
    host_id,
    name,
    email,
    phone,
    is_verified,
    is_active,
    rating,
    country,
    joined_at
FROM samples.wanderbricks.hosts;

num_affected_rows,num_inserted_rows
19384,19384


In [0]:
%sql

SELECT COUNT(*) AS total_registros
FROM wanderbricks_analitica.bronze.hosts;

total_registros
58152


In [0]:
%sql

CREATE TABLE IF NOT EXISTS wanderbricks_analitica.bronze.page_views (
    event STRING,
    property_id BIGINT,
    timestamp TIMESTAMP,
    user_id BIGINT
)
USING DELTA;

In [0]:
%sql

INSERT INTO wanderbricks_analitica.bronze.page_views
SELECT
    'page_view' AS event,
    property_id,
    timestamp,
    user_id
FROM samples.wanderbricks.page_views;

num_affected_rows,num_inserted_rows
500000,500000


In [0]:
%sql

SELECT
    event,
    COUNT(*) AS cantidad
FROM wanderbricks_analitica.bronze.page_views
GROUP BY event;

event,cantidad
page_view,1500000


In [0]:
%sql

SHOW TABLES IN wanderbricks_analitica.bronze;

database,tableName,isTemporary
bronze,bookings,false
bronze,clickstream,false
bronze,countries,false
bronze,destinations,false
bronze,hosts,false
bronze,page_views,false
bronze,payments,false
bronze,properties,false
bronze,reviews,false
bronze,users,false


## Creacion de la tabla silver para calidad y estandarizacion de datos
como tal esta tabla la realizamos para funciones de transformacion y calidad, asi que aqui asociaremos todos los datos que no denoten un valor para su devida consulta garantizando asi una mejor trazabilidad de los datos.

In [0]:
%sql

CREATE TABLE IF NOT EXISTS wanderbricks_analitica.silver.bookings (
    booking_id BIGINT,
    user_id BIGINT,
    property_id BIGINT,
    check_in DATE,
    check_out DATE,
    guests_count INT,
    total_amount DECIMAL(15,4),
    status STRING,
    created_at TIMESTAMP,
    updated_at TIMESTAMP
)
USING DELTA;

In [0]:
%sql

INSERT INTO wanderbricks_analitica.silver.bookings
SELECT
    booking_id,
    user_id,
    property_id,
    check_in,
    check_out,
    guests_count,
    CAST(total_amount AS DECIMAL(15,4)) AS total_amount,
    status,
    created_at,
    updated_at
FROM wanderbricks_analitica.bronze.bookings
WHERE total_amount >= 0;

num_affected_rows,num_inserted_rows
216738,216738


In [0]:
%sql

SELECT COUNT(*) AS total_registros
FROM wanderbricks_analitica.silver.bookings;

total_registros
433476


In [0]:
%sql

SELECT COUNT(*) AS montos_negativos
FROM wanderbricks_analitica.silver.bookings
WHERE total_amount < 0;

montos_negativos
0


## 4. Construcción de la capa Silver

Una vez finalizada la ingestión de los datos originales en la capa Bronze, se inició la construcción de la capa Silver.

En esta etapa se aplican reglas de transformación y validación orientadas a mejorar la calidad y consistencia de los datos antes de utilizarlos para el análisis. A diferencia de Bronze, donde se conserva la información de origen, Silver contiene datos preparados para su utilización analítica.

Entre las transformaciones aplicadas se incluyen la conversión de tipos de datos, la eliminación de registros que incumplen reglas de calidad y la validación de las relaciones entre las entidades.

La primera tabla construida en esta capa corresponde a `bookings`, donde se transforma `total_amount` al tipo `DECIMAL(15,4)` y se excluyen los registros con montos negativos identificados durante la exploración inicial.


In [0]:
%sql

CREATE TABLE IF NOT EXISTS wanderbricks_analitica.silver.payments (
    payment_id BIGINT,
    booking_id BIGINT,
    amount DECIMAL(15,4),
    payment_method STRING,
    status STRING,
    payment_date TIMESTAMP
)
USING DELTA;

In [0]:
%sql

INSERT INTO wanderbricks_analitica.silver.payments
SELECT
    payment_id,
    booking_id,
    CAST(amount AS DECIMAL(15,4)) AS amount,
    payment_method,
    status,
    payment_date
FROM wanderbricks_analitica.bronze.payments;

num_affected_rows,num_inserted_rows
49638,49638


In [0]:
%sql

SELECT COUNT(*) AS total_registros
FROM wanderbricks_analitica.silver.payments;

total_registros
148914


In [0]:
%sql

SELECT COUNT(*) AS pagos_sin_reserva
FROM wanderbricks_analitica.silver.payments p
LEFT JOIN wanderbricks_analitica.silver.bookings b
    ON p.booking_id = b.booking_id
WHERE b.booking_id IS NULL;

pagos_sin_reserva
27


In [0]:
%sql

SELECT
    p.payment_id,
    p.booking_id,
    p.amount,
    p.payment_method,
    p.status,
    p.payment_date
FROM wanderbricks_analitica.silver.payments p
LEFT JOIN wanderbricks_analitica.silver.bookings b
    ON p.booking_id = b.booking_id
WHERE b.booking_id IS NULL
ORDER BY p.booking_id;

payment_id,booking_id,amount,payment_method,status,payment_date
17570289932442,28499,-42.7600,credit_card,completed,2025-06-13T23:22:26.000Z
17570289932442,28499,-42.7600,credit_card,completed,2025-06-13T23:22:26.000Z
17570289932442,28499,-42.7600,credit_card,completed,2025-06-13T23:22:26.000Z
17570290012307,72653,965.6000,paypal,completed,2024-08-31T17:54:11.000Z
17570290012307,72653,965.6000,paypal,completed,2024-08-31T17:54:11.000Z
17570290012307,72653,965.6000,paypal,completed,2024-08-31T17:54:11.000Z
17570289881126,72655,603.8000,credit_card,completed,2025-07-30T22:37:17.000Z
17570289881126,72655,603.8000,credit_card,completed,2025-07-30T22:37:17.000Z
17570289881126,72655,603.8000,credit_card,completed,2025-07-30T22:37:17.000Z
17570289946976,72657,433.2400,credit_card,completed,2025-05-11T19:49:35.000Z


In [0]:
%sql

SELECT
    payment_id,
    COUNT(*) AS cantidad
FROM wanderbricks_analitica.silver.payments
GROUP BY payment_id
HAVING COUNT(*) > 1
ORDER BY cantidad DESC;

payment_id,cantidad
17570283856242,12
17570283830924,12
17570283829000,12
17570283828299,12
17570283855816,12
17570283869775,12
17570283845718,12
17570283834124,12
17570283846716,12
17570283861972,12


In [0]:
%sql

SELECT
    COUNT(*) AS ids_duplicados,
    SUM(cantidad - 1) AS registros_excedentes
FROM (
    SELECT
        payment_id,
        COUNT(*) AS cantidad
    FROM wanderbricks_analitica.silver.payments
    GROUP BY payment_id
    HAVING COUNT(*) > 1
);

ids_duplicados,registros_excedentes
45173,103741


In [0]:
%sql

SELECT
    cantidad AS registros_por_payment_id,
    COUNT(*) AS cantidad_de_payment_ids
FROM (
    SELECT
        payment_id,
        COUNT(*) AS cantidad
    FROM wanderbricks_analitica.silver.payments
    GROUP BY payment_id
)
GROUP BY cantidad
ORDER BY cantidad;

registros_por_payment_id,cantidad_de_payment_ids
3,41066
6,3765
9,326
12,16


In [0]:
%sql

CREATE TABLE IF NOT EXISTS wanderbricks_analitica.silver.reviews (
    booking_id BIGINT,
    comment STRING,
    created_at TIMESTAMP,
    is_deleted BOOLEAN,
    property_id BIGINT,
    rating DOUBLE,
    review_id BIGINT,
    updated_at TIMESTAMP,
    user_id BIGINT
)
USING DELTA;

In [0]:
%sql

INSERT INTO wanderbricks_analitica.silver.reviews
SELECT
    booking_id,
    comment,
    created_at,
    is_deleted,
    property_id,
    rating,
    review_id,
    updated_at,
    user_id
FROM wanderbricks_analitica.bronze.reviews;

num_affected_rows,num_inserted_rows
299379,299379


In [0]:
%sql

SELECT COUNT(*) AS total_registros
FROM wanderbricks_analitica.silver.reviews;

total_registros
598758


In [0]:
%sql

SELECT
    COUNT(*) AS total_registros,
    SUM(CASE WHEN rating IS NULL THEN 1 ELSE 0 END) AS ratings_nulos,
    SUM(CASE WHEN rating < 1 OR rating > 5 THEN 1 ELSE 0 END) AS ratings_fuera_de_rango
FROM wanderbricks_analitica.silver.reviews;

total_registros,ratings_nulos,ratings_fuera_de_rango
598758,2856,0


In [0]:
%sql

SELECT
    is_deleted,
    COUNT(*) AS cantidad
FROM wanderbricks_analitica.silver.reviews
GROUP BY is_deleted
ORDER BY is_deleted;

is_deleted,cantidad
false,595902
true,2856


In [0]:
%sql

SELECT
    is_deleted,
    COUNT(*) AS `total_reseñas`,
    SUM(CASE WHEN rating IS NULL THEN 1 ELSE 0 END) AS ratings_nulos
FROM wanderbricks_analitica.silver.reviews
GROUP BY is_deleted
ORDER BY is_deleted;

is_deleted,total_reseñas,ratings_nulos
false,595902,0
true,2856,2856


In [0]:
%sql

CREATE TABLE IF NOT EXISTS wanderbricks_analitica.silver.properties (
    property_id BIGINT,
    host_id BIGINT,
    destination_id BIGINT,
    title STRING,
    description STRING,
    base_price DECIMAL(15,4),
    property_type STRING,
    max_guests INT,
    bedrooms INT,
    bathrooms INT,
    property_latitude DECIMAL(10,6),
    property_longitude DECIMAL(10,6),
    created_at DATE
)
USING DELTA;

In [0]:
%sql

INSERT INTO wanderbricks_analitica.silver.properties
SELECT
    property_id,
    host_id,
    destination_id,
    title,
    description,
    CAST(base_price AS DECIMAL(15,4)) AS base_price,
    property_type,
    max_guests,
    bedrooms,
    bathrooms,
    CAST(property_latitude AS DECIMAL(10,6)) AS property_latitude,
    CAST(property_longitude AS DECIMAL(10,6)) AS property_longitude,
    created_at
FROM wanderbricks_analitica.bronze.properties;

num_affected_rows,num_inserted_rows
54489,54489


In [0]:
%sql

SELECT COUNT(*) AS total_registros
FROM wanderbricks_analitica.silver.properties;

total_registros
108978


In [0]:
%sql

SELECT
    COUNT(*) AS total_registros,
    SUM(CASE WHEN base_price IS NULL THEN 1 ELSE 0 END) AS precios_nulos,
    SUM(CASE WHEN base_price < 0 THEN 1 ELSE 0 END) AS precios_negativos,
    SUM(CASE WHEN property_latitude IS NULL THEN 1 ELSE 0 END) AS latitudes_nulas,
    SUM(CASE WHEN property_longitude IS NULL THEN 1 ELSE 0 END) AS longitudes_nulas,
    SUM(CASE WHEN property_latitude < -90 OR property_latitude > 90 THEN 1 ELSE 0 END) AS latitudes_invalidas,
    SUM(CASE WHEN property_longitude < -180 OR property_longitude > 180 THEN 1 ELSE 0 END) AS longitudes_invalidas
FROM wanderbricks_analitica.silver.properties;

total_registros,precios_nulos,precios_negativos,latitudes_nulas,longitudes_nulas,latitudes_invalidas,longitudes_invalidas
108978,0,0,0,0,0,0


In [0]:
%sql

SELECT
    COUNT(*) AS total_propiedades,
    SUM(CASE WHEN h.host_id IS NULL THEN 1 ELSE 0 END) AS hosts_sin_correspondencia,
    SUM(CASE WHEN d.destination_id IS NULL THEN 1 ELSE 0 END) AS destinos_sin_correspondencia
FROM wanderbricks_analitica.silver.properties p
LEFT JOIN wanderbricks_analitica.bronze.hosts h
    ON p.host_id = h.host_id
LEFT JOIN wanderbricks_analitica.bronze.destinations d
    ON p.destination_id = d.destination_id;

total_propiedades,hosts_sin_correspondencia,destinos_sin_correspondencia
980802,0,0


In [0]:
%sql

CREATE TABLE IF NOT EXISTS wanderbricks_analitica.silver.destinations (
    destination_id BIGINT,
    destination STRING,
    country STRING,
    state_or_province STRING,
    state_or_province_code STRING,
    description STRING
)
USING DELTA;

In [0]:
%sql

INSERT INTO wanderbricks_analitica.silver.destinations
SELECT
    destination_id,
    destination,
    country,
    state_or_province,
    state_or_province_code,
    description
FROM wanderbricks_analitica.bronze.destinations;

num_affected_rows,num_inserted_rows
126,126


In [0]:
%sql

SELECT COUNT(*) AS total_registros
FROM wanderbricks_analitica.silver.destinations;

total_registros
252


In [0]:
%sql

SELECT
    COUNT(*) AS total_destinos,
    SUM(CASE WHEN destination IS NULL THEN 1 ELSE 0 END) AS destinos_nulos,
    SUM(CASE WHEN country IS NULL THEN 1 ELSE 0 END) AS paises_nulos,
    SUM(CASE WHEN description IS NULL THEN 1 ELSE 0 END) AS descripciones_nulas
FROM wanderbricks_analitica.silver.destinations;

total_destinos,destinos_nulos,paises_nulos,descripciones_nulas
252,0,0,0


In [0]:
%sql

CREATE TABLE IF NOT EXISTS wanderbricks_analitica.silver.clickstream (
    event STRING,
    metadata STRUCT<
        device: STRING,
        referrer: STRING
    >,
    property_id BIGINT,
    timestamp TIMESTAMP,
    user_id BIGINT
)
USING DELTA;

In [0]:
%sql

INSERT INTO wanderbricks_analitica.silver.clickstream
SELECT
    event,
    metadata,
    property_id,
    timestamp,
    user_id
FROM wanderbricks_analitica.bronze.clickstream;

num_affected_rows,num_inserted_rows
300000,300000


In [0]:
%sql

SELECT COUNT(*) AS total_registros
FROM wanderbricks_analitica.silver.clickstream;

total_registros
600000


In [0]:
%sql

SELECT
    event,
    metadata.device AS dispositivo,
    metadata.referrer AS origen,
    property_id,
    user_id
FROM wanderbricks_analitica.silver.clickstream
LIMIT 10;

event,dispositivo,origen,property_id,user_id
view,desktop,email,5135,16390
search,desktop,direct,7805,3655
click,mobile,direct,17349,100094
filter,tablet,ad,6128,102187
search,mobile,google,3438,40863
search,mobile,email,14475,115938
search,tablet,ad,4788,69029
view,desktop,direct,12099,37389
view,mobile,google,6600,534
click,mobile,google,13528,107117


In [0]:
%sql

CREATE TABLE IF NOT EXISTS wanderbricks_analitica.silver.users (
    user_id BIGINT,
    email STRING,
    name STRING,
    country STRING,
    user_type STRING,
    created_at TIMESTAMP,
    is_business BOOLEAN,
    company_name STRING
)
USING DELTA;

In [0]:
%sql

INSERT INTO wanderbricks_analitica.silver.users
SELECT
    user_id,
    email,
    name,
    country,
    user_type,
    created_at,
    is_business,
    company_name
FROM wanderbricks_analitica.bronze.users;

num_affected_rows,num_inserted_rows
373527,373527


In [0]:
%sql

SELECT COUNT(*) AS total_registros
FROM wanderbricks_analitica.silver.users;

total_registros
747054


In [0]:
%sql

SELECT
    is_business,
    COUNT(*) AS cantidad,
    SUM(CASE WHEN company_name IS NULL THEN 1 ELSE 0 END) AS empresa_sin_nombre
FROM wanderbricks_analitica.silver.users
GROUP BY is_business
ORDER BY is_business;

is_business,cantidad,empresa_sin_nombre
false,371802,371802
true,375252,0


In [0]:
%sql

CREATE TABLE IF NOT EXISTS wanderbricks_analitica.silver.hosts (
    host_id BIGINT,
    name STRING,
    email STRING,
    phone STRING,
    is_verified BOOLEAN,
    is_active BOOLEAN,
    rating FLOAT,
    country STRING,
    joined_at DATE
)
USING DELTA;

In [0]:
%sql

INSERT INTO wanderbricks_analitica.silver.hosts
SELECT
    host_id,
    name,
    email,
    phone,
    is_verified,
    is_active,
    rating,
    country,
    joined_at
FROM wanderbricks_analitica.bronze.hosts;

num_affected_rows,num_inserted_rows
58152,58152


In [0]:
%sql

SELECT COUNT(*) AS total_registros
FROM wanderbricks_analitica.silver.hosts;

total_registros
116304


In [0]:
%sql

SELECT
    COUNT(*) AS total_hosts,
    SUM(CASE WHEN rating IS NULL THEN 1 ELSE 0 END) AS ratings_nulos,
    SUM(CASE WHEN rating < 1 OR rating > 5 THEN 1 ELSE 0 END) AS ratings_fuera_de_rango
FROM wanderbricks_analitica.silver.hosts;

total_hosts,ratings_nulos,ratings_fuera_de_rango
116304,0,0


In [0]:
%sql

CREATE TABLE IF NOT EXISTS wanderbricks_analitica.silver.countries (
    country STRING,
    country_code STRING,
    continent STRING
)
USING DELTA;

In [0]:
%sql

INSERT INTO wanderbricks_analitica.silver.countries
SELECT
    country,
    country_code,
    continent
FROM wanderbricks_analitica.bronze.countries;

num_affected_rows,num_inserted_rows
504,504


In [0]:
%sql

SELECT COUNT(*) AS total_registros
FROM wanderbricks_analitica.silver.countries;

total_registros
1008


In [0]:
%sql

CREATE TABLE IF NOT EXISTS wanderbricks_analitica.silver.page_views (
    event STRING,
    property_id BIGINT,
    timestamp TIMESTAMP,
    user_id BIGINT
)
USING DELTA;

In [0]:
%sql

INSERT INTO wanderbricks_analitica.silver.page_views
SELECT
    event,
    property_id,
    timestamp,
    user_id
FROM wanderbricks_analitica.bronze.page_views;

num_affected_rows,num_inserted_rows
1500000,1500000


In [0]:
%sql

SELECT COUNT(*) AS total_registros
FROM wanderbricks_analitica.silver.page_views;

total_registros
3000000


In [0]:
%sql

SHOW TABLES IN wanderbricks_analitica.silver;

database,tableName,isTemporary
silver,bookings,false
silver,clickstream,false
silver,countries,false
silver,destinations,false
silver,hosts,false
silver,page_views,false
silver,payments,false
silver,properties,false
silver,reviews,false
silver,users,false


### 4.1 Resultado de la construcción de la capa Silver

La capa Silver fue construida a partir de las tablas almacenadas previamente en Bronze. Durante este proceso se aplicaron transformaciones de tipos de datos, reglas básicas de calidad y validaciones de integridad referencial.

En `bookings` se excluyeron los registros con `total_amount` negativo, corrigiendo la anomalía identificada durante la exploración inicial. En `payments` se conservaron los registros que presentan valores repetidos de `payment_id`, debido a que el análisis demostró que estos registros pueden corresponder a diferentes reservas, montos, métodos de pago y fechas.

En `properties` se estandarizaron los tipos de datos de precios y coordenadas geográficas mediante conversiones a `DECIMAL`, mientras que las relaciones con anfitriones y destinos fueron verificadas mediante consultas de integridad referencial.

Las tablas `reviews`, `destinations`, `users`, `hosts`, `countries` y `page_views` fueron trasladadas a Silver y sometidas a validaciones de volumen y calidad según sus características.

En el caso de `clickstream`, se conservó la estructura semiestructurada `metadata` como un `STRUCT` compuesto por `device` y `referrer`, permitiendo mantener la flexibilidad del dato original y consultar sus atributos internos.

Finalmente, la consulta de las tablas disponibles en el esquema `silver` permitió verificar la creación de las estructuras correspondientes a esta capa. Con esto se dispone de un conjunto de datos preparado para las siguientes etapas de análisis y construcción de la capa analítica.


In [0]:
%sql

CREATE SCHEMA IF NOT EXISTS wanderbricks_analitica.gold;

SHOW TABLES IN wanderbricks_analitica.gold;

database,tableName,isTemporary
gold,demo_acid,false
gold,resumen_metodos_pago,false
gold,resumen_rendimiento_hosts,false
gold,resumen_reservas_destino,false
gold,resumen_usuarios,false


In [0]:
%sql

SELECT
    d.destination,
    COUNT(b.booking_id) AS total_reservas,
    SUM(b.total_amount) AS ingresos_totales,
    AVG(b.total_amount) AS ingreso_promedio
FROM wanderbricks_analitica.silver.bookings b
JOIN wanderbricks_analitica.silver.properties p
    ON b.property_id = p.property_id
JOIN wanderbricks_analitica.silver.destinations d
    ON p.destination_id = d.destination_id
WHERE b.total_amount >= 0
GROUP BY d.destination
ORDER BY ingresos_totales DESC;

destination,total_reservas,ingresos_totales,ingreso_promedio
Phuket,1575072,876571102.8000,556.52763988
Gold Coast,1405512,779780334.2832,554.80161982
Mallorca,1401192,770178281.0616,549.65934794
Paris,1219536,680010640.6032,557.59784098
Abu Dhabi,953424,528326919.4248,554.13637524
Tokyo,778248,432655274.8584,555.93496528
Dubai,669600,367655593.6800,549.06749355
Osaka,661392,366778031.0616,554.55468325
Singapore,648864,360565536.2400,555.68738016
New York,632664,353260284.4368,558.36950488


### 5.1 Resumen analítico de reservas por destino

Como primera estructura de la capa Gold se construyó la tabla `resumen_reservas_destino`. Esta tabla integra información de reservas, propiedades y destinos para generar indicadores agregados orientados al análisis del desempeño de cada destino.

Los indicadores calculados incluyen el número total de reservas, los ingresos totales y el ingreso promedio por reserva. La información se materializa como una tabla Delta en la capa Gold, permitiendo disponer de resultados analíticos listos para su consulta sin necesidad de repetir las operaciones de unión y agregación sobre las tablas de Silver.


In [0]:
%sql

CREATE OR REPLACE TABLE wanderbricks_analitica.gold.resumen_reservas_destino
USING DELTA
AS
SELECT
    d.destination,
    COUNT(b.booking_id) AS total_reservas,
    SUM(b.total_amount) AS ingresos_totales,
    AVG(b.total_amount) AS ingreso_promedio
FROM wanderbricks_analitica.silver.bookings b
JOIN wanderbricks_analitica.silver.properties p
    ON b.property_id = p.property_id
JOIN wanderbricks_analitica.silver.destinations d
    ON p.destination_id = d.destination_id
WHERE b.total_amount >= 0
GROUP BY d.destination;

num_affected_rows,num_inserted_rows


In [0]:
%sql

SELECT *
FROM wanderbricks_analitica.gold.resumen_reservas_destino
ORDER BY ingresos_totales DESC;

destination,total_reservas,ingresos_totales,ingreso_promedio
Phuket,1575072,876571102.8000,556.52763988
Gold Coast,1405512,779780334.2832,554.80161982
Mallorca,1401192,770178281.0616,549.65934794
Paris,1219536,680010640.6032,557.59784098
Abu Dhabi,953424,528326919.4248,554.13637524
Tokyo,778248,432655274.8584,555.93496528
Dubai,669600,367655593.6800,549.06749355
Osaka,661392,366778031.0616,554.55468325
Singapore,648864,360565536.2400,555.68738016
New York,632664,353260284.4368,558.36950488


In [0]:
%sql

SHOW TABLES IN wanderbricks_analitica.gold;

database,tableName,isTemporary
gold,demo_acid,false
gold,resumen_metodos_pago,false
gold,resumen_rendimiento_hosts,false
gold,resumen_reservas_destino,false
gold,resumen_usuarios,false


In [0]:
%sql

SELECT
    payment_method,
    status,
    COUNT(*) AS total_pagos,
    SUM(amount) AS monto_total,
    AVG(amount) AS monto_promedio
FROM wanderbricks_analitica.silver.payments
WHERE amount >= 0
GROUP BY payment_method, status
ORDER BY monto_total DESC;

payment_method,status,total_pagos,monto_total,monto_promedio
credit_card,completed,27513,15261127.1100,554.68786065
apple_pay,completed,27171,15209891.5791,559.78401896
paypal,completed,27477,15170560.3794,552.11851292
bank_transfer,completed,27000,15135263.0103,560.56529668
google_pay,completed,27123,15127114.9506,557.72277958
bank_transfer,failed,309,170857.9500,552.93834951
paypal,failed,300,170653.1100,568.84370000
apple_pay,failed,297,159136.0500,535.81161616
google_pay,failed,270,146710.7700,543.37322222
credit_card,failed,276,143036.5500,518.24836957


In [0]:
%sql

CREATE OR REPLACE TABLE wanderbricks_analitica.gold.resumen_metodos_pago
USING DELTA
AS
SELECT
    payment_method,
    status,
    COUNT(*) AS total_pagos,
    SUM(amount) AS monto_total,
    AVG(amount) AS monto_promedio
FROM wanderbricks_analitica.silver.payments
WHERE amount >= 0
GROUP BY payment_method, status;

num_affected_rows,num_inserted_rows


### 5.2 Resumen de pagos por método y estado

Se construyó la tabla `resumen_metodos_pago` con el propósito de analizar el comportamiento de los pagos realizados en la plataforma.

La tabla agrupa los registros por método de pago y estado de la transacción, generando indicadores como el número de pagos, el monto total procesado y el monto promedio por pago.

Esta estructura permite comparar los diferentes métodos de pago y analizar su participación según el estado de las transacciones, facilitando la generación de indicadores para el análisis financiero de la plataforma.


In [0]:
%sql

SELECT *
FROM wanderbricks_analitica.gold.resumen_metodos_pago
ORDER BY status, monto_total DESC;

payment_method,status,total_pagos,monto_total,monto_promedio
credit_card,completed,27513,15261127.1100,554.68786065
apple_pay,completed,27171,15209891.5791,559.78401896
paypal,completed,27477,15170560.3794,552.11851292
bank_transfer,completed,27000,15135263.0103,560.56529668
google_pay,completed,27123,15127114.9506,557.72277958
bank_transfer,failed,309,170857.9500,552.93834951
paypal,failed,300,170653.1100,568.84370000
apple_pay,failed,297,159136.0500,535.81161616
google_pay,failed,270,146710.7700,543.37322222
credit_card,failed,276,143036.5500,518.24836957


In [0]:
%sql

SHOW TABLES IN wanderbricks_analitica.gold;

database,tableName,isTemporary
gold,demo_acid,false
gold,resumen_metodos_pago,false
gold,resumen_rendimiento_hosts,false
gold,resumen_reservas_destino,false
gold,resumen_usuarios,false


In [0]:
%sql

SELECT
    h.host_id,
    h.name AS nombre_host,
    h.country,
    h.rating AS rating_host,
    COUNT(b.booking_id) AS total_reservas,
    SUM(b.total_amount) AS ingresos_totales,
    AVG(b.total_amount) AS ingreso_promedio
FROM wanderbricks_analitica.silver.hosts h
JOIN wanderbricks_analitica.silver.properties p
    ON h.host_id = p.host_id
JOIN wanderbricks_analitica.silver.bookings b
    ON p.property_id = b.property_id
WHERE b.total_amount >= 0
GROUP BY
    h.host_id,
    h.name,
    h.country,
    h.rating
ORDER BY ingresos_totales DESC;

host_id,nombre_host,country,rating_host,total_reservas,ingresos_totales,ingreso_promedio
10895,Holly Chapman,United Arab Emirates,4.58,100440,54763421.7600,545.23518280
12672,Heather Smith,United Arab Emirates,3.53,93312,51627399.8616,553.27717616
11652,Tina Ford,United Arab Emirates,3.55,85752,47792369.5200,557.33241814
12769,Amber Gutierrez,United Arab Emirates,4.32,82296,45432239.0400,552.05889764
11610,William Booker,United Arab Emirates,3.81,80136,43220662.5600,539.34140162
11900,Bethany Hall,United Arab Emirates,4.55,73008,40711738.3416,557.63393521
14106,Oscar Buchanan,United Arab Emirates,3.0,70200,39007077.1416,555.65636954
8136,Paul Acosta,United Arab Emirates,4.46,67824,37954671.1200,559.60531847
14601,Tracy Murray,United Arab Emirates,3.88,68040,37235654.6400,547.26123810
10613,David Underwood,United Arab Emirates,3.64,69768,37123708.3200,532.10222910


In [0]:
%sql

SELECT
    h.host_id,
    h.name AS nombre_host,
    h.country,
    h.rating AS rating_host,
    COUNT(b.booking_id) AS total_reservas,
    SUM(b.total_amount) AS ingresos_totales,
    AVG(b.total_amount) AS ingreso_promedio
FROM wanderbricks_analitica.silver.hosts h
JOIN wanderbricks_analitica.silver.properties p
    ON h.host_id = p.host_id
JOIN wanderbricks_analitica.silver.bookings b
    ON p.property_id = b.property_id
WHERE b.total_amount >= 0
GROUP BY
    h.host_id,
    h.name,
    h.country,
    h.rating
ORDER BY ingresos_totales DESC
LIMIT 10;

host_id,nombre_host,country,rating_host,total_reservas,ingresos_totales,ingreso_promedio
10895,Holly Chapman,United Arab Emirates,4.58,100440,54763421.7600,545.23518280
12672,Heather Smith,United Arab Emirates,3.53,93312,51627399.8616,553.27717616
11652,Tina Ford,United Arab Emirates,3.55,85752,47792369.5200,557.33241814
12769,Amber Gutierrez,United Arab Emirates,4.32,82296,45432239.0400,552.05889764
11610,William Booker,United Arab Emirates,3.81,80136,43220662.5600,539.34140162
11900,Bethany Hall,United Arab Emirates,4.55,73008,40711738.3416,557.63393521
14106,Oscar Buchanan,United Arab Emirates,3.0,70200,39007077.1416,555.65636954
8136,Paul Acosta,United Arab Emirates,4.46,67824,37954671.1200,559.60531847
14601,Tracy Murray,United Arab Emirates,3.88,68040,37235654.6400,547.26123810
10613,David Underwood,United Arab Emirates,3.64,69768,37123708.3200,532.10222910


### 5.3 Resumen del rendimiento de los anfitriones

Se construyó la tabla `resumen_rendimiento_hosts` para analizar el desempeño de los anfitriones dentro de la plataforma.

La tabla integra información de `hosts`, `properties` y `bookings`, permitiendo relacionar cada anfitrión con sus propiedades y las reservas asociadas. A partir de esta integración se calculan indicadores como el número total de reservas, los ingresos totales generados y el ingreso promedio por reserva.

También se conserva la valoración registrada para cada anfitrión y su país de residencia, lo que permite complementar los indicadores económicos con información descriptiva.

Esta estructura facilita la identificación de anfitriones con mayor volumen de reservas y generación de ingresos, y proporciona una base para posteriores análisis comparativos del rendimiento de los anfitriones.


In [0]:
%sql

CREATE OR REPLACE TABLE wanderbricks_analitica.gold.resumen_rendimiento_hosts
USING DELTA
AS
SELECT
    h.host_id,
    h.name AS nombre_host,
    h.country,
    h.rating AS rating_host,
    COUNT(b.booking_id) AS total_reservas,
    SUM(b.total_amount) AS ingresos_totales,
    AVG(b.total_amount) AS ingreso_promedio
FROM wanderbricks_analitica.silver.hosts h
JOIN wanderbricks_analitica.silver.properties p
    ON h.host_id = p.host_id
JOIN wanderbricks_analitica.silver.bookings b
    ON p.property_id = b.property_id
WHERE b.total_amount >= 0
GROUP BY
    h.host_id,
    h.name,
    h.country,
    h.rating;

num_affected_rows,num_inserted_rows


In [0]:
%sql

SELECT COUNT(*) AS total_hosts_analizados
FROM wanderbricks_analitica.gold.resumen_rendimiento_hosts;

total_hosts_analizados
3742


In [0]:
%sql

SELECT
    u.user_id,
    u.name AS nombre_usuario,
    u.country,
    u.user_type,
    u.is_business,
    COUNT(b.booking_id) AS total_reservas,
    SUM(b.total_amount) AS gasto_total,
    AVG(b.total_amount) AS gasto_promedio
FROM wanderbricks_analitica.silver.users u
JOIN wanderbricks_analitica.silver.bookings b
    ON u.user_id = b.user_id
WHERE b.total_amount >= 0
GROUP BY
    u.user_id,
    u.name,
    u.country,
    u.user_type,
    u.is_business
ORDER BY gasto_total DESC
LIMIT 10;

user_id,nombre_usuario,country,user_type,is_business,total_reservas,gasto_total,gasto_promedio
53131,Kim-Campbell,Ecuador,business,true,180,160877.5200,893.76400000
52885,Erik Wilkins,Bangladesh,individual,false,180,151519.3200,841.77400000
103472,"Park, Gomez and Douglas",Uganda,business,true,180,150556.6800,836.42600000
36206,Kramer-Hernandez,Qatar,business,true,180,138607.9200,770.04400000
46878,Jennifer Wolfe,China,individual,false,180,134798.4000,748.88000000
15574,Tyler-Williams,China,business,true,180,130917.6000,727.32000000
49393,Roberts and Sons,Japan,business,true,180,129328.2000,718.49000000
37954,Miller-Parker,China,business,true,144,128593.4400,893.01000000
10961,"Lee, Rogers and Ellis",Iran,business,true,144,126043.9200,875.30500000
110964,Watson Inc,United States,business,true,180,125540.2800,697.44600000


### 5.4 Resumen del comportamiento de los usuarios

Se construyó la tabla `resumen_usuarios` con el propósito de analizar el comportamiento de los usuarios a partir de sus reservas.

La tabla integra información de `users` y `bookings`, permitiendo relacionar las características de cada usuario con su actividad de reserva. Se calculan indicadores como el número total de reservas, el gasto total y el gasto promedio por reserva.

Además, se conservan atributos como el país de residencia, el tipo de usuario y la identificación de usuarios empresariales. Esta información permite segmentar a los usuarios y analizar diferentes patrones de comportamiento y consumo dentro de la plataforma.

La tabla constituye una estructura analítica que facilita la identificación de usuarios con mayor actividad y gasto, así como la comparación entre diferentes segmentos de usuarios.


In [0]:
%sql

CREATE OR REPLACE TABLE wanderbricks_analitica.gold.resumen_usuarios
USING DELTA
AS
SELECT
    u.user_id,
    u.name AS nombre_usuario,
    u.country,
    u.user_type,
    u.is_business,
    COUNT(b.booking_id) AS total_reservas,
    SUM(b.total_amount) AS gasto_total,
    AVG(b.total_amount) AS gasto_promedio
FROM wanderbricks_analitica.silver.users u
JOIN wanderbricks_analitica.silver.bookings b
    ON u.user_id = b.user_id
WHERE b.total_amount >= 0
GROUP BY
    u.user_id,
    u.name,
    u.country,
    u.user_type,
    u.is_business;

num_affected_rows,num_inserted_rows


In [0]:
%sql

SELECT COUNT(*) AS usuarios_con_reservas
FROM wanderbricks_analitica.gold.resumen_usuarios;

usuarios_con_reservas
54708


In [0]:
%sql

SHOW TABLES IN wanderbricks_analitica.gold;

database,tableName,isTemporary
gold,demo_acid,false
gold,resumen_metodos_pago,false
gold,resumen_rendimiento_hosts,false
gold,resumen_reservas_destino,false
gold,resumen_usuarios,false


In [0]:
%sql

DESCRIBE HISTORY wanderbricks_analitica.gold.resumen_reservas_destino;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
6,2026-08-29T22:27:32.000Z,72971641301131,brandon.cano@est.iudigital.edu.co,CREATE OR REPLACE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.enableDeletionVectors"":""true"",""delta.parquet.format.version"":""2.12.0"",""io.unitycatalog.tableId"":""0c13a7bd-72f4-4d95-bd28-aa3b9f0d0afe""}, statsOnLoad -> true)",null,List(9725776144894),a5a5dad1-8cf6-4430-b337-4c638aabd08c,0829-222253-xb1sadhu-v2n,5,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 1, numRemovedBytes -> 2916, numDeletionVectorsRemoved -> 0, numOutputRows -> 41, numOutputBytes -> 2577)",null,Databricks-Runtime/19.2.x-aarch64-photon-scala2.13
5,2026-08-29T21:09:33.000Z,72971641301131,brandon.cano@est.iudigital.edu.co,UPDATE,Map(predicate -> []),null,List(9725776144894),92834295-dd2e-40e0-bbe5-e07e091631a3,0829-210409-34ij1kno-v2n,4,WriteSerializable,false,"Map(numRemovedFiles -> 1, numRemovedBytes -> 2533, numCopiedRows -> 0, numDeletionVectorsAdded -> 0, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 2169, numDeletionVectorsUpdated -> 0, scanTimeMs -> 31, numAddedFiles -> 1, numUpdatedRows -> 41, numAddedBytes -> 2916, rewriteTimeMs -> 2111)",null,Databricks-Runtime/19.2.x-aarch64-photon-scala2.13
4,2026-08-29T21:09:23.000Z,72971641301131,brandon.cano@est.iudigital.edu.co,ADD COLUMNS,"Map(columns -> [{""column"":{""name"":""categoria_destino"",""type"":""string"",""nullable"":true,""metadata"":{}}}])",null,List(9725776144894),c70d53c2-52da-4653-8961-345b2baa8b76,0829-210409-34ij1kno-v2n,3,WriteSerializable,true,Map(),null,Databricks-Runtime/19.2.x-aarch64-photon-scala2.13
3,2026-08-29T21:08:47.000Z,72971641301131,brandon.cano@est.iudigital.edu.co,CREATE OR REPLACE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.enableDeletionVectors"":""true"",""delta.parquet.format.version"":""2.12.0"",""io.unitycatalog.tableId"":""0c13a7bd-72f4-4d95-bd28-aa3b9f0d0afe""}, statsOnLoad -> true)",null,List(9725776144894),f2c0f3e4-0eee-46d2-bffa-014eb8ca5dcf,0829-210409-34ij1kno-v2n,2,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 1, numRemovedBytes -> 2878, numDeletionVectorsRemoved -> 0, numOutputRows -> 41, numOutputBytes -> 2533)",null,Databricks-Runtime/19.2.x-aarch64-photon-scala2.13
2,2026-08-27T00:12:37.000Z,72971641301131,brandon.cano@est.iudigital.edu.co,UPDATE,Map(predicate -> []),null,List(9725776144894),f14f8091-c6f7-4899-88c4-c07774c14de0,0826-224657-dzjhn111-v2n,1,WriteSerializable,false,"Map(numRemovedFiles -> 1, numRemovedBytes -> 2497, numCopiedRows -> 0, numDeletionVectorsAdded -> 0, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 2667, numDeletionVectorsUpdated -> 0, scanTimeMs -> 39, numAddedFiles -> 1, numUpdatedRows -> 41, numAddedBytes -> 2878, rewriteTimeMs -> 2598)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
1,2026-08-27T00:10:09.000Z,72971641301131,brandon.cano@est.iudigital.edu.co,ADD COLUMNS,"Map(columns -> [{""column"":{""name"":""categoria_destino"",""type"":""string"",""nullable"":true,""metadata"":{}}}])",null,List(9725776144894),2283f7f2-6aed-4b3f-bcbe-6cb6356ae485,0826-224657-dzjhn111-v2n,0,WriteSerializable,true,Map(),null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
0,2026-08-26T23:53:00.000Z,72971641301131,brandon.cano@est.iudigital.edu.co,CREATE OR REPLACE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""

In [0]:
%sql

SELECT COUNT(*) AS registros_version_0
FROM wanderbricks_analitica.gold.resumen_reservas_destino VERSION AS OF 0;

registros_version_0
41


In [0]:
%sql

ALTER TABLE wanderbricks_analitica.gold.resumen_reservas_destino
ADD COLUMNS (categoria_destino STRING);

In [0]:
%sql

DESCRIBE TABLE wanderbricks_analitica.gold.resumen_reservas_destino;

col_name,data_type,comment
destination,string,null
total_reservas,bigint,null
ingresos_totales,"decimal(25,4)",null
ingreso_promedio,"decimal(19,8)",null
categoria_destino,string,null


In [0]:
%sql

SELECT *
FROM wanderbricks_analitica.gold.resumen_reservas_destino VERSION AS OF 0
LIMIT 5;

destination,total_reservas,ingresos_totales,ingreso_promedio
Byron Bay,930,505398.2500,543.43897849
Mykonos,687,378590.4900,551.07786026
Whistler,56,32451.4000,579.48928571
Kitzbühel,18,11267.1500,625.95277778
Santorini,666,371458.6201,557.74567583


In [0]:
%sql

SELECT *
FROM wanderbricks_analitica.gold.resumen_reservas_destino
LIMIT 5;

destination,total_reservas,ingresos_totales,ingreso_promedio,categoria_destino
Beijing,15552,9395956.8000,604.16388889,null
Cairo,222480,121236341.8032,544.93141767,null
Sardinia,378000,211687421.7168,560.01963417,null
Berlin,480600,266840924.4000,555.22456180,null
San Francisco,540432,295162410.2616,546.16012794,null


In [0]:
%sql

SELECT *
FROM wanderbricks_analitica.gold.resumen_reservas_destino
LIMIT 5;

destination,total_reservas,ingresos_totales,ingreso_promedio,categoria_destino
Beijing,15552,9395956.8000,604.16388889,null
Cairo,222480,121236341.8032,544.93141767,null
Sardinia,378000,211687421.7168,560.01963417,null
Berlin,480600,266840924.4000,555.22456180,null
San Francisco,540432,295162410.2616,546.16012794,null


In [0]:
%sql

UPDATE wanderbricks_analitica.gold.resumen_reservas_destino
SET categoria_destino =
    CASE
        WHEN total_reservas >= 5000 THEN 'Alta demanda'
        WHEN total_reservas >= 1000 THEN 'Demanda media'
        ELSE 'Baja demanda'
    END;

num_affected_rows
41


### 5.5 Evolución de esquema y categorización de destinos

Como parte de la evolución de la tabla `resumen_reservas_destino`, se agregó la columna `categoria_destino` mediante una modificación del esquema de la tabla Delta.

Posteriormente, se aplicó una regla de clasificación basada en el número total de reservas de cada destino. Los destinos con 5.000 o más reservas fueron clasificados como de **Alta demanda**, aquellos con entre 1.000 y 4.999 reservas como de **Demanda media**, y los destinos con menos de 1.000 reservas como de **Baja demanda**.

Esta transformación permite incorporar una dimensión categórica al resumen de reservas y facilita la segmentación de los destinos según su nivel de demanda.

El proceso constituye un ejemplo práctico de evolución de esquema y transformación de datos sobre una tabla Delta, demostrando la flexibilidad del enfoque Lakehouse para adaptar las estructuras analíticas a nuevas necesidades.


In [0]:
%sql

SELECT
    categoria_destino,
    COUNT(*) AS cantidad_destinos,
    SUM(total_reservas) AS total_reservas
FROM wanderbricks_analitica.gold.resumen_reservas_destino
GROUP BY categoria_destino
ORDER BY total_reservas DESC;

categoria_destino,cantidad_destinos,total_reservas
Alta demanda,37,15596928
Demanda media,3,7776
Baja demanda,1,432


In [0]:
%sql

DESCRIBE HISTORY wanderbricks_analitica.gold.resumen_reservas_destino;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
8,2026-08-29T22:28:24.000Z,72971641301131,brandon.cano@est.iudigital.edu.co,UPDATE,Map(predicate -> []),null,List(9725776144894),c227b4c5-0b4b-4bb0-9a7f-3f37b4f26365,0829-222253-xb1sadhu-v2n,7,WriteSerializable,false,"Map(numRemovedFiles -> 1, numRemovedBytes -> 2577, numCopiedRows -> 0, numDeletionVectorsAdded -> 0, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 2319, numDeletionVectorsUpdated -> 0, scanTimeMs -> 26, numAddedFiles -> 1, numUpdatedRows -> 41, numAddedBytes -> 2945, rewriteTimeMs -> 2267)",null,Databricks-Runtime/19.2.x-aarch64-photon-scala2.13
7,2026-08-29T22:28:12.000Z,72971641301131,brandon.cano@est.iudigital.edu.co,ADD COLUMNS,"Map(columns -> [{""column"":{""name"":""categoria_destino"",""type"":""string"",""nullable"":true,""metadata"":{}}}])",null,List(9725776144894),abc241d2-6d3a-41ac-bfc2-a44e9ad01382,0829-222253-xb1sadhu-v2n,6,WriteSerializable,true,Map(),null,Databricks-Runtime/19.2.x-aarch64-photon-scala2.13
6,2026-08-29T22:27:32.000Z,72971641301131,brandon.cano@est.iudigital.edu.co,CREATE OR REPLACE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.enableDeletionVectors"":""true"",""delta.parquet.format.version"":""2.12.0"",""io.unitycatalog.tableId"":""0c13a7bd-72f4-4d95-bd28-aa3b9f0d0afe""}, statsOnLoad -> true)",null,List(9725776144894),a5a5dad1-8cf6-4430-b337-4c638aabd08c,0829-222253-xb1sadhu-v2n,5,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 1, numRemovedBytes -> 2916, numDeletionVectorsRemoved -> 0, numOutputRows -> 41, numOutputBytes -> 2577)",null,Databricks-Runtime/19.2.x-aarch64-photon-scala2.13
5,2026-08-29T21:09:33.000Z,72971641301131,brandon.cano@est.iudigital.edu.co,UPDATE,Map(predicate -> []),null,List(9725776144894),92834295-dd2e-40e0-bbe5-e07e091631a3,0829-210409-34ij1kno-v2n,4,WriteSerializable,false,"Map(numRemovedFiles -> 1, numRemovedBytes -> 2533, numCopiedRows -> 0, numDeletionVectorsAdded -> 0, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 2169, numDeletionVectorsUpdated -> 0, scanTimeMs -> 31, numAddedFiles -> 1, numUpdatedRows -> 41, numAddedBytes -> 2916, rewriteTimeMs -> 2111)",null,Databricks-Runtime/19.2.x-aarch64-photon-scala2.13
4,2026-08-29T21:09:23.000Z,72971641301131,brandon.cano@est.iudigital.edu.co,ADD COLUMNS,"Map(columns -> [{""column"":{""name"":""categoria_destino"",""type"":""string"",""nullable"":true,""metadata"":{}}}])",null,List(9725776144894),c70d53c2-52da-4653-8961-345b2baa8b76,0829-210409-34ij1kno-v2n,3,WriteSerializable,true,Map(),null,Databricks-Runtime/19.2.x-aarch64-photon-scala2.13
3,2026-08-29T21:08:47.000Z,72971641301131,brandon.cano@est.iudigital.edu.co,CREATE OR REPLACE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.enableDeletionVectors"":""true"",""delta.parquet.format.version"":""2.12.0"",""io.unitycatalog.tableId"":""0c13a7bd-72f4-4d95-bd28-aa3b9f0d0afe""}, statsOnLoad -> true)",null,List(9725776144894),f2c0f3e4-0eee-46d2-bffa-014eb8ca5dcf,0829-210409-34ij1kno-v2n,2,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 1, numRemovedBytes -> 2878, numDeletionVectorsRemoved -> 0, numOutputRows -> 41, numOutputBytes -> 2533)",null,Databricks-Runtime/19.2.x-aarch64-photon-scala2.13
2,2026-08-27T00:12:37.000Z,72971641301131,brandon.cano@est.iudigital.edu.co,UPDATE,Map(predicate -> []),null,List(9725776144894),f14f8091-c6f7-4899-88c4-c07774c14de0,0826-224657-dzjhn111-v2n,1,WriteSerializable,false,"Map(numRemovedFiles -> 1, numRe

In [0]:
%sql

SELECT
    categoria_destino,
    COUNT(*) AS cantidad
FROM wanderbricks_analitica.gold.resumen_reservas_destino VERSION AS OF 1
GROUP BY categoria_destino;

categoria_destino,cantidad
null,41


In [0]:
%sql

DESCRIBE TABLE wanderbricks_analitica.bronze.clickstream;

col_name,data_type,comment
event,string,null
metadata,struct,null
property_id,bigint,null
timestamp,timestamp,null
user_id,bigint,null


In [0]:
%sql

SELECT
    event,
    metadata,
    property_id,
    timestamp,
    user_id
FROM wanderbricks_analitica.bronze.clickstream
LIMIT 10;

event,metadata,property_id,timestamp,user_id
view,"List(desktop, email)",5135,2025-09-04T23:26:54.935Z,16390
search,"List(desktop, direct)",7805,2025-09-04T23:26:54.935Z,3655
click,"List(mobile, direct)",17349,2025-09-04T23:26:54.935Z,100094
filter,"List(tablet, ad)",6128,2025-09-04T23:26:54.935Z,102187
search,"List(mobile, google)",3438,2025-09-04T23:26:54.935Z,40863
search,"List(mobile, email)",14475,2025-09-04T23:26:54.935Z,115938
search,"List(tablet, ad)",4788,2025-09-04T23:26:54.935Z,69029
view,"List(desktop, direct)",12099,2025-09-04T23:26:54.935Z,37389
view,"List(mobile, google)",6600,2025-09-04T23:26:54.935Z,534
click,"List(mobile, google)",13528,2025-09-04T23:26:54.935Z,107117


In [0]:
%sql

SELECT
    event,
    metadata.device AS dispositivo,
    metadata.referrer AS origen,
    property_id,
    user_id
FROM wanderbricks_analitica.bronze.clickstream
LIMIT 10;

event,dispositivo,origen,property_id,user_id
view,desktop,email,5135,16390
search,desktop,direct,7805,3655
click,mobile,direct,17349,100094
filter,tablet,ad,6128,102187
search,mobile,google,3438,40863
search,mobile,email,14475,115938
search,tablet,ad,4788,69029
view,desktop,direct,12099,37389
view,mobile,google,6600,534
click,mobile,google,13528,107117


In [0]:
%sql

SELECT
    metadata.device AS dispositivo,
    metadata.referrer AS origen,
    COUNT(*) AS total_interacciones
FROM wanderbricks_analitica.bronze.clickstream
GROUP BY
    metadata.device,
    metadata.referrer
ORDER BY total_interacciones DESC;

dispositivo,origen,total_interacciones
tablet,ad,25497
tablet,direct,25437
tablet,email,25419
desktop,email,25374
desktop,google,25092
tablet,google,24906
desktop,ad,24786
mobile,google,24744
mobile,direct,24708
mobile,ad,24702


## 6. Demostración de las capacidades ACID de Delta Lake

Además del modelo analítico desarrollado para Wanderbricks, se realizó una demostración práctica de las propiedades transaccionales de las tablas Delta.

Para ello se creó una tabla de prueba denominada `demo_acid`, independiente de las tablas analíticas del modelo. Esta tabla permite realizar operaciones de inserción, actualización y eliminación sobre datos almacenados en formato Delta, con el propósito de evidenciar el comportamiento transaccional de la tecnología.

La demostración complementa la implementación del Lakehouse al mostrar de forma práctica una de las principales características que justifican el uso de Delta Lake frente a un almacenamiento de archivos sin control transaccional.


In [0]:
%sql

CREATE OR REPLACE TABLE wanderbricks_analitica.gold.demo_acid (
    id INT,
    descripcion STRING
)
USING DELTA;

In [0]:
%sql

INSERT INTO wanderbricks_analitica.gold.demo_acid
VALUES
    (1, 'Operacion A'),
    (2, 'Operacion B');

num_affected_rows,num_inserted_rows
2,2


In [0]:
%sql

SELECT *
FROM wanderbricks_analitica.gold.demo_acid;

id,descripcion
1,Operacion A
2,Operacion B


In [0]:
%sql

DESCRIBE HISTORY wanderbricks_analitica.gold.demo_acid;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
5,2026-08-29T22:28:41.000Z,72971641301131,brandon.cano@est.iudigital.edu.co,WRITE,"Map(mode -> Append, statsOnLoad -> true, partitionBy -> [])",null,List(9725776144894),4aec406b-b07e-4793-b1a7-6bd373c1a2df,0829-222253-xb1sadhu-v2n,4,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 2, numOutputBytes -> 1093)",null,Databricks-Runtime/19.2.x-aarch64-photon-scala2.13
4,2026-08-29T22:28:40.000Z,72971641301131,brandon.cano@est.iudigital.edu.co,CREATE OR REPLACE TABLE,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.enableDeletionVectors"":""true"",""delta.parquet.format.version"":""2.12.0"",""delta.rowTracking.materializedRowCommitVersionColumnName"":""_row-commit-version-col-ea8d6b28-fced-4047-adf0-45d122296608"",""delta.rowTracking.materializedRowIdColumnName"":""_row-id-col-d7805aea-b049-4017-9cfc-8020f606646b""}, statsOnLoad -> false)",null,List(9725776144894),0a8b0ebb-ffd7-4774-a866-ab63236c4729,0829-222253-xb1sadhu-v2n,3,WriteSerializable,false,Map(),null,Databricks-Runtime/19.2.x-aarch64-photon-scala2.13
3,2026-08-29T21:09:48.000Z,72971641301131,brandon.cano@est.iudigital.edu.co,WRITE,"Map(mode -> Append, statsOnLoad -> true, partitionBy -> [])",null,List(9725776144894),81f174f8-aacb-48d1-9d43-ee282ba8b8de,0829-210409-34ij1kno-v2n,2,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 2, numOutputBytes -> 1093)",null,Databricks-Runtime/19.2.x-aarch64-photon-scala2.13
2,2026-08-29T21:09:46.000Z,72971641301131,brandon.cano@est.iudigital.edu.co,CREATE OR REPLACE TABLE,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.enableDeletionVectors"":""true"",""delta.parquet.format.version"":""2.12.0"",""delta.rowTracking.materializedRowCommitVersionColumnName"":""_row-commit-version-col-ea8d6b28-fced-4047-adf0-45d122296608"",""delta.rowTracking.materializedRowIdColumnName"":""_row-id-col-d7805aea-b049-4017-9cfc-8020f606646b""}, statsOnLoad -> false)",null,List(9725776144894),dca64564-512d-4b6a-a791-3b489cee5402,0829-210409-34ij1kno-v2n,1,WriteSerializable,false,Map(),null,Databricks-Runtime/19.2.x-aarch64-photon-scala2.13
1,2026-08-27T02:10:19.000Z,72971641301131,brandon.cano@est.iudigital.edu.co,WRITE,"Map(mode -> Append, statsOnLoad -> true, partitionBy -> [])",null,List(9725776144894),ab459816-beb3-471f-8783-5acc90046d33,0827-020407-soe2fxzb-v2n,0,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 2, numOutputBytes -> 1093)",null,Databricks-Runtime/19.2.x-aarch64-photon-scala2.13
0,2026-08-27T02:08:39.000Z,72971641301131,brandon.cano@est.iudigital.edu.co,CREATE OR REPLACE TABLE,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.enableDeletionVectors"":""true"",""delta.parquet.format.version"":""2.12.0"",""delta.enableRowTracking"":""true"",""delta.rowTracking.materializedRowCommitVersionColumnName"":""_row-commit-version-col-ea8d6b28-fced-4047-adf0-45d122296608"",""delta.rowTracking.materializedRowIdColumnName"":""_row-id-col-d7805aea-b049-4017-9cfc-8020f606646b""}, statsOnLoad -> false)",null,List(9725776144894),4e179c67-d073-4396-8a54-49ab60f16a6f,0827-020407-soe2fxzb-v2n,null,WriteSerializable,true,Map(),null,Databricks-Runtime/19.2.x-aarch64-photon-scala2.13


Primera consulta en pyspark


In [0]:
from pyspark.sql import functions as F

clickstream = spark.table(
    "wanderbricks_analitica.bronze.clickstream"
)

resultado = (
    clickstream
    .groupBy(
        F.col("metadata.device").alias("dispositivo"),
        F.col("metadata.referrer").alias("origen")
    )
    .agg(
        F.count("*").alias("total_interacciones")
    )
    .orderBy(
        F.col("total_interacciones").desc()
    )
)

display(resultado)

dispositivo,origen,total_interacciones
tablet,ad,25497
tablet,direct,25437
tablet,email,25419
desktop,email,25374
desktop,google,25092
tablet,google,24906
desktop,ad,24786
mobile,google,24744
mobile,direct,24708
mobile,ad,24702


In [0]:
%sql

SELECT
    payment_method,
    status,
    COUNT(*) AS total_transacciones,
    SUM(amount) AS ingresos_totales,
    AVG(amount) AS ingreso_promedio
FROM wanderbricks_analitica.silver.payments
GROUP BY
    payment_method,
    status
ORDER BY
    status,
    ingresos_totales DESC;

payment_method,status,total_transacciones,ingresos_totales,ingreso_promedio
credit_card,completed,27516,15260998.8300,554.62272242
apple_pay,completed,27171,15209891.5791,559.78401896
paypal,completed,27477,15170560.3794,552.11851292
bank_transfer,completed,27000,15135263.0103,560.56529668
google_pay,completed,27126,15127092.1806,557.66025881
paypal,failed,315,164844.7500,523.31666667
bank_transfer,failed,330,158847.4500,481.35590909
apple_pay,failed,312,152548.8000,488.93846154
google_pay,failed,294,129889.3200,441.80040816
credit_card,failed,300,128194.6200,427.31540000


In [0]:
from pyspark.sql import functions as F

payments = spark.table(
    "wanderbricks_analitica.silver.payments"
)

resultado = (
    payments
    .groupBy("payment_method", "status")
    .agg(
        F.count("*").alias("total_transacciones"),
        F.sum("amount").alias("ingresos_totales"),
        F.avg("amount").alias("ingreso_promedio")
    )
    .orderBy(
        "status",
        F.col("ingresos_totales").desc()
    )
)

display(resultado)

payment_method,status,total_transacciones,ingresos_totales,ingreso_promedio
credit_card,completed,27516,15260998.8300,554.62272242
apple_pay,completed,27171,15209891.5791,559.78401896
paypal,completed,27477,15170560.3794,552.11851292
bank_transfer,completed,27000,15135263.0103,560.56529668
google_pay,completed,27126,15127092.1806,557.66025881
paypal,failed,315,164844.7500,523.31666667
bank_transfer,failed,330,158847.4500,481.35590909
apple_pay,failed,312,152548.8000,488.93846154
google_pay,failed,294,129889.3200,441.80040816
credit_card,failed,300,128194.6200,427.31540000


In [0]:
%sql

SELECT
    destination,
    total_reservas,
    ingresos_totales,
    ingreso_promedio
FROM wanderbricks_analitica.gold.resumen_reservas_destino
ORDER BY ingresos_totales DESC
LIMIT 10;

destination,total_reservas,ingresos_totales,ingreso_promedio
Phuket,1575072,876571102.8000,556.52763988
Gold Coast,1405512,779780334.2832,554.80161982
Mallorca,1401192,770178281.0616,549.65934794
Paris,1219536,680010640.6032,557.59784098
Abu Dhabi,953424,528326919.4248,554.13637524
Tokyo,778248,432655274.8584,555.93496528
Dubai,669600,367655593.6800,549.06749355
Osaka,661392,366778031.0616,554.55468325
Singapore,648864,360565536.2400,555.68738016
New York,632664,353260284.4368,558.36950488


In [0]:
from pyspark.sql import functions as F

reservas_destino = spark.table(
    "wanderbricks_analitica.gold.resumen_reservas_destino"
)

resultado = (
    reservas_destino
    .select(
        "destination",
        "total_reservas",
        "ingresos_totales",
        "ingreso_promedio"
    )
    .orderBy(
        F.col("ingresos_totales").desc()
    )
    .limit(10)
)

display(resultado)

destination,total_reservas,ingresos_totales,ingreso_promedio
Phuket,1575072,876571102.8000,556.52763988
Gold Coast,1405512,779780334.2832,554.80161982
Mallorca,1401192,770178281.0616,549.65934794
Paris,1219536,680010640.6032,557.59784098
Abu Dhabi,953424,528326919.4248,554.13637524
Tokyo,778248,432655274.8584,555.93496528
Dubai,669600,367655593.6800,549.06749355
Osaka,661392,366778031.0616,554.55468325
Singapore,648864,360565536.2400,555.68738016
New York,632664,353260284.4368,558.36950488


In [0]:
%sql

SELECT
    host_id,
    nombre_host,
    country,
    rating_host,
    total_reservas,
    ingresos_totales,
    ingreso_promedio
FROM wanderbricks_analitica.gold.resumen_rendimiento_hosts
ORDER BY ingresos_totales DESC
LIMIT 10;

host_id,nombre_host,country,rating_host,total_reservas,ingresos_totales,ingreso_promedio
10895,Holly Chapman,United Arab Emirates,4.58,100440,54763421.7600,545.23518280
12672,Heather Smith,United Arab Emirates,3.53,93312,51627399.8616,553.27717616
11652,Tina Ford,United Arab Emirates,3.55,85752,47792369.5200,557.33241814
12769,Amber Gutierrez,United Arab Emirates,4.32,82296,45432239.0400,552.05889764
11610,William Booker,United Arab Emirates,3.81,80136,43220662.5600,539.34140162
11900,Bethany Hall,United Arab Emirates,4.55,73008,40711738.3416,557.63393521
14106,Oscar Buchanan,United Arab Emirates,3.0,70200,39007077.1416,555.65636954
8136,Paul Acosta,United Arab Emirates,4.46,67824,37954671.1200,559.60531847
14601,Tracy Murray,United Arab Emirates,3.88,68040,37235654.6400,547.26123810
10613,David Underwood,United Arab Emirates,3.64,69768,37123708.3200,532.10222910


In [0]:
from pyspark.sql import functions as F

hosts = spark.table(
    "wanderbricks_analitica.gold.resumen_rendimiento_hosts"
)

resultado = (
    hosts
    .select(
        "host_id",
        "nombre_host",
        "country",
        "rating_host",
        "total_reservas",
        "ingresos_totales",
        "ingreso_promedio"
    )
    .orderBy(
        F.col("ingresos_totales").desc()
    )
    .limit(10)
)

display(resultado)

host_id,nombre_host,country,rating_host,total_reservas,ingresos_totales,ingreso_promedio
10895,Holly Chapman,United Arab Emirates,4.58,100440,54763421.7600,545.23518280
12672,Heather Smith,United Arab Emirates,3.53,93312,51627399.8616,553.27717616
11652,Tina Ford,United Arab Emirates,3.55,85752,47792369.5200,557.33241814
12769,Amber Gutierrez,United Arab Emirates,4.32,82296,45432239.0400,552.05889764
11610,William Booker,United Arab Emirates,3.81,80136,43220662.5600,539.34140162
11900,Bethany Hall,United Arab Emirates,4.55,73008,40711738.3416,557.63393521
14106,Oscar Buchanan,United Arab Emirates,3.0,70200,39007077.1416,555.65636954
8136,Paul Acosta,United Arab Emirates,4.46,67824,37954671.1200,559.60531847
14601,Tracy Murray,United Arab Emirates,3.88,68040,37235654.6400,547.26123810
10613,David Underwood,United Arab Emirates,3.64,69768,37123708.3200,532.10222910


In [0]:
%sql

SELECT
    categoria_destino,
    destination,
    total_reservas,
    ingresos_totales,
    ingreso_promedio
FROM wanderbricks_analitica.gold.resumen_reservas_destino
ORDER BY
    CASE categoria_destino
        WHEN 'Alta demanda' THEN 1
        WHEN 'Demanda media' THEN 2
        WHEN 'Baja demanda' THEN 3
        ELSE 4
    END,
    total_reservas DESC;

categoria_destino,destination,total_reservas,ingresos_totales,ingreso_promedio
Alta demanda,Phuket,1575072,876571102.8000,556.52763988
Alta demanda,Gold Coast,1405512,779780334.2832,554.80161982
Alta demanda,Mallorca,1401192,770178281.0616,549.65934794
Alta demanda,Paris,1219536,680010640.6032,557.59784098
Alta demanda,Abu Dhabi,953424,528326919.4248,554.13637524
Alta demanda,Tokyo,778248,432655274.8584,555.93496528
Alta demanda,Dubai,669600,367655593.6800,549.06749355
Alta demanda,Osaka,661392,366778031.0616,554.55468325
Alta demanda,Singapore,648864,360565536.2400,555.68738016
Alta demanda,New York,632664,353260284.4368,558.36950488


In [0]:
from pyspark.sql import functions as F

reservas_destino = spark.table(
    "wanderbricks_analitica.gold.resumen_reservas_destino"
)

resultado = (
    reservas_destino
    .select(
        "categoria_destino",
        "destination",
        "total_reservas",
        "ingresos_totales",
        "ingreso_promedio"
    )
    .withColumn(
        "orden_categoria",
        F.when(F.col("categoria_destino") == "Alta demanda", 1)
         .when(F.col("categoria_destino") == "Demanda media", 2)
         .when(F.col("categoria_destino") == "Baja demanda", 3)
         .otherwise(4)
    )
    .orderBy(
        "orden_categoria",
        F.col("total_reservas").desc()
    )
    .drop("orden_categoria")
)

display(resultado)

categoria_destino,destination,total_reservas,ingresos_totales,ingreso_promedio
Alta demanda,Phuket,1575072,876571102.8000,556.52763988
Alta demanda,Gold Coast,1405512,779780334.2832,554.80161982
Alta demanda,Mallorca,1401192,770178281.0616,549.65934794
Alta demanda,Paris,1219536,680010640.6032,557.59784098
Alta demanda,Abu Dhabi,953424,528326919.4248,554.13637524
Alta demanda,Tokyo,778248,432655274.8584,555.93496528
Alta demanda,Dubai,669600,367655593.6800,549.06749355
Alta demanda,Osaka,661392,366778031.0616,554.55468325
Alta demanda,Singapore,648864,360565536.2400,555.68738016
Alta demanda,New York,632664,353260284.4368,558.36950488


In [0]:
%sql

SHOW VOLUMES IN wanderbricks_analitica.bronze;

database,volume_name
bronze,raw_data


# 1. Arquitectura del entorno de datos

La arquitectura propuesta para Wanderbricks se basa en un enfoque Lakehouse implementado sobre Databricks y Delta Lake. El flujo comprende las fuentes de datos, la ingesta, el almacenamiento organizado en las capas Bronze, Silver y Gold, el procesamiento y finalmente el consumo de la información para análisis y toma de decisiones.

La arquitectura busca separar las responsabilidades de ingestión, transformación y consumo, manteniendo una organización clara de los datos y aplicando mecanismos de gobierno y control de acceso.


## 1.1 Diagrama de arquitectura

```text
┌──────────────────────────────┐
│       FUENTES DE DATOS       │
│                              │
│ • Wanderbricks               │
│ • Usuarios                   │
│ • Propiedades                │
│ • Reservas                   │
│ • Pagos                      │
│ • Reseñas                    │
│ • Clickstream / Page Views   │
└──────────────┬───────────────┘
               │
               ▼
┌──────────────────────────────┐
│            INGESTA           │
│                              │
│ Carga de datos hacia         │
│ el entorno Databricks        │
└──────────────┬───────────────┘
               │
               ▼
┌──────────────────────────────┐
│          BRONZE              │
│                              │
│ Datos ingeridos              │
│ Conservación de datos        │
│ originales                   │
└──────────────┬───────────────┘
               │
               ▼
┌──────────────────────────────┐
│           SILVER             │
│                              │
│ Limpieza                     │
│ Validación                   │
│ Transformación               │
│ Estandarización              │
└──────────────┬───────────────┘
               │
               ▼
┌──────────────────────────────┐
│            GOLD              │
│                              │
│ Datos preparados para       │
│ análisis y consumo           │
└──────────────┬───────────────┘
               │
               ▼
┌──────────────────────────────┐
│          CONSUMO             │
│                              │
│ • Consultas SQL              │
│ • Análisis                   │
│ • Indicadores                │
│ • Toma de decisiones         │
└──────────────────────────────┘
```


## 1.2 Responsabilidades de administración

En el entorno administrado de Databricks existe una separación entre las responsabilidades del proveedor de la plataforma y las responsabilidades del equipo que utiliza el entorno.

**Responsabilidades principalmente gestionadas por el proveedor:**

* Infraestructura física y centros de datos.
* Servidores y recursos físicos subyacentes.
* Mantenimiento de la plataforma Databricks.
* Actualizaciones y componentes administrados de la plataforma.
* Parte de la infraestructura necesaria para ejecutar los servicios de procesamiento.

**Responsabilidades gestionadas por el equipo:**

* Organización de catálogos, esquemas y objetos de datos.
* Diseño de las capas Bronze, Silver y Gold.
* Definición de tablas y estructuras analíticas.
* Procesos de ingestión y transformación.
* Administración de permisos y acceso a los datos.
* Configuración de Jobs y procesos de automatización.
* Definición de roles y políticas de acceso.
* Gobierno y calidad de los datos.

Esta separación permite que el equipo se concentre principalmente en la organización, procesamiento y gobierno de los datos, mientras que la infraestructura subyacente es administrada en gran parte por el proveedor de la plataforma.


 Verificamos primero si hay catalogos existentes y cuales antes de crear nuestro devido catalogo

In [0]:
%sql

SHOW CATALOGS;

catalog
samples
system
wanderbricks_analitica
workspace


Ahora vamos a comprobar los schemas

In [0]:
%sql

SHOW SCHEMAS IN wanderbricks_analitica;

databaseName
bronze
default
gold
information_schema
silver


In [0]:
%sql

SHOW VOLUMES IN wanderbricks_analitica.bronze;

database,volume_name
bronze,raw_data


## 2. Organización del entorno: catálogos, esquemas y volúmenes

Para organizar el entorno analítico se utiliza el catálogo **`wanderbricks_analitica`**, construido a partir de la arquitectura desarrollada en la actividad anterior. Esta organización permite separar los datos de acuerdo con su nivel de procesamiento y facilita la aplicación de mecanismos de gobernanza, seguridad y control de acceso.

La estructura propuesta para el entorno es la siguiente:

```text
wanderbricks_analitica
│
├── bronze
│   └── raw_data (Volume)
│
├── silver
│
├── gold
│
├── default
│
└── information_schema
```

### Justificación de la estructura

* **Catálogo `wanderbricks_analitica`:** funciona como contenedor principal de los objetos analíticos del proyecto Wanderbricks.
* **Esquema `bronze`:** contiene los datos en su estado inicial o con un nivel mínimo de transformación. Su propósito es conservar la información proveniente de las fuentes de origen.
* **Volume `raw_data`:** proporciona un espacio administrado para almacenar archivos de datos en bruto asociados al proceso de ingestión.
* **Esquema `silver`:** está destinado a los datos que han pasado por procesos de limpieza, validación, estandarización y transformación.
* **Esquema `gold`:** contiene los datos preparados para análisis, generación de indicadores y consumo por parte de los usuarios finales.
* **`default` e `information_schema`:** corresponden a esquemas disponibles dentro del entorno de Databricks y no hacen parte directamente de las capas principales de procesamiento diseñadas para Wanderbricks.

Esta organización mantiene la separación entre las diferentes etapas del procesamiento de datos y facilita posteriormente la implementación de permisos diferenciados, trazabilidad y mecanismos de gobernanza sobre cada capa.


In [0]:
%sql
SHOW GROUPS;

name,directGroup
admins,null
users,null
users-clone-2026-08-25-1623-UTC (created by Databricks),null


In [0]:
%sql
SHOW TABLES IN wanderbricks_analitica.bronze;

database,tableName,isTemporary
bronze,bookings,false
bronze,clickstream,false
bronze,countries,false
bronze,destinations,false
bronze,hosts,false
bronze,page_views,false
bronze,payments,false
bronze,properties,false
bronze,reviews,false
bronze,users,false


In [0]:
%sql
SHOW TABLES IN wanderbricks_analitica.gold;

database,tableName,isTemporary
gold,demo_acid,false
gold,resumen_metodos_pago,false
gold,resumen_rendimiento_hosts,false
gold,resumen_reservas_destino,false
gold,resumen_usuarios,false


In [0]:
%sql
SHOW GRANTS ON TABLE wanderbricks_analitica.gold.resumen_reservas_destino;

Principal,ActionType,ObjectType,ObjectKey


In [0]:
%sql
SHOW GRANTS ON TABLE wanderbricks_analitica.bronze.bookings;

Principal,ActionType,ObjectType,ObjectKey


In [0]:
%sql
SHOW GRANTS ON TABLE wanderbricks_analitica.bronze.bookings;

Principal,ActionType,ObjectType,ObjectKey


In [0]:
%sql
SHOW GRANTS ON TABLE wanderbricks_analitica.gold.resumen_reservas_destino;

Principal,ActionType,ObjectType,ObjectKey


In [0]:
%sql
GRANT MODIFY ON TABLE wanderbricks_analitica.bronze.bookings TO `account users`;

In [0]:
%sql
GRANT SELECT ON TABLE wanderbricks_analitica.gold.resumen_reservas_destino TO `account users`;

In [0]:
%sql
SHOW GRANTS ON TABLE wanderbricks_analitica.bronze.bookings;

Principal,ActionType,ObjectType,ObjectKey
account users,MODIFY,TABLE,wanderbricks_analitica.bronze.bookings


In [0]:
%sql
SHOW GRANTS ON TABLE wanderbricks_analitica.gold.resumen_reservas_destino;

Principal,ActionType,ObjectType,ObjectKey
account users,SELECT,TABLE,wanderbricks_analitica.gold.resumen_reservas_destino


Matriz de roles

La matriz diferencia las responsabilidades según el nivel de acceso requerido. El analista trabaja principalmente sobre la información procesada de Silver y Gold; el ingeniero de datos requiere permisos de lectura y escritura para realizar procesos de ingestión y transformación; y el administrador mantiene el control global del entorno, incluyendo seguridad, permisos y configuración.

    | Rol                    | Bronze            | Silver            | Gold                 | Administración |
| ---------------------- | ----------------- | ----------------- | -------------------- | -------------- |
| **Analista**           | Lectura limitada  | Lectura           | Lectura y consulta   | No             |
| **Ingeniero de datos** | Lectura/escritura | Lectura/escritura | Escritura controlada | No             |
| **Administrador**      | Control total     | Control total     | Control total        | Sí             |


![Screenshot 2026-09-20 182056.png](./Screenshot 2026-09-20 182056.png "Screenshot 2026-09-20 182056.png")

## 5. Lineage y trazabilidad de los datos

Para evidenciar la trazabilidad de los datos se utilizó **Catalog Explorer** sobre la tabla `wanderbricks_analitica.silver.bookings`.

La vista de **Lineage** permite identificar tanto los objetos **upstream**, relacionados con el origen y procesamiento de la información, como los objetos **downstream**, que utilizan posteriormente los datos de la tabla.

En la evidencia se observa como origen la tabla `wanderbricks_analitica.bronze.bookings` y el notebook `EA1_Diseño_Base_Datos_Analitica`. También se identifican como objetos downstream diferentes tablas de la capa Gold, entre ellas `resumen_reservas_destino`, `resumen_rendimiento_hosts` y `resumen_usuarios`.

Esta trazabilidad permite conocer el flujo de los datos entre las diferentes capas de la arquitectura **Bronze → Silver → Gold**, facilitando el seguimiento del origen, transformación y consumo de la información dentro del entorno analítico.



## 6. Automatización mediante Databricks Jobs

Para automatizar el procesamiento del Lakehouse de Wanderbricks se propone un Job en Databricks Workflows compuesto por dos tareas encadenadas.

La primera tarea, `01_validar_bronze`, verifica que la tabla `wanderbricks_analitica.bronze.bookings` contenga registros.

La segunda tarea, `02_validar_silver`, verifica que la tabla `wanderbricks_analitica.silver.bookings` contenga registros. Esta tarea depende de la finalización exitosa de `01_validar_bronze`.

El Job se programa para ejecutarse diariamente a las 02:00 a. m., utilizando la zona horaria `America/Bogota`.

La estructura de ejecución es:

`01_validar_bronze` → `02_validar_silver`

La evidencia de ejecución se obtiene desde Databricks Workflows, donde se debe mostrar que ambas tareas finalizaron exitosamente.


![Screenshot 2026-09-20 215254.png](./Screenshot 2026-09-20 215254.png "Screenshot 2026-09-20 215254.png")

In [0]:
7.## Diseño de una alternativa equivalente sobre IaaS

Para implementar una solución equivalente utilizando infraestructura como servicio (IaaS), se propone desplegar máquinas virtuales y servicios administrados directamente por el equipo responsable de la plataforma.

| Componente             | Propuesta                                                              |
| ---------------------- | ---------------------------------------------------------------------- |
| Servidores             | 3 máquinas virtuales                                                   |
| Nodo principal         | 8 vCPU, 32 GB RAM                                                      |
| Nodos de procesamiento | 2 máquinas de 4 vCPU y 16 GB RAM cada una                              |
| Sistema operativo      | Ubuntu Server LTS                                                      |
| Procesamiento          | Apache Spark                                                           |
| Lenguaje               | Python / PySpark y SQL                                                 |
| Almacenamiento         | Disco SSD para sistema y almacenamiento de datos                       |
| Base de datos          | PostgreSQL para metadatos y servicios auxiliares                       |
| Red                    | Red virtual privada, subredes, reglas de firewall y acceso restringido |
| Seguridad              | Usuarios, grupos, llaves SSH, reglas de acceso y cifrado               |
| Monitoreo              | Logs, métricas de CPU, memoria, almacenamiento y procesos              |



Distribución propuesta

La arquitectura IaaS tendría un nodo principal encargado de coordinar los procesos y dos nodos destinados al procesamiento distribuido. Los datos podrían almacenarse en discos asociados a las máquinas virtuales o en un servicio de almacenamiento de objetos.

El flujo conservaría la lógica implementada en la actividad anterior:

Fuentes → Ingesta → Bronze → Silver → Gold → Consumo

La principal diferencia sería que, en IaaS, el equipo tendría que instalar, configurar, actualizar y mantener directamente el sistema operativo, Spark, servicios de almacenamiento, seguridad, redes, monitoreo y demás componentes necesarios.

Esfuerzo de implementación y operación

El despliegue inicial requeriría configurar las máquinas virtuales, establecer la red, instalar el sistema operativo y las dependencias, configurar Spark, establecer los mecanismos de almacenamiento y seguridad y posteriormente realizar pruebas de funcionamiento.

Durante la operación también sería necesario administrar actualizaciones, capacidad de almacenamiento, rendimiento, fallos de las máquinas, copias de seguridad, monitoreo y controles de seguridad.

Por lo tanto, aunque IaaS proporciona un mayor nivel de control sobre la infraestructura, también transfiere al equipo una mayor cantidad de responsabilidades operativas que en una plataforma PaaS como Databricks son gestionadas por el proveedor.

8. ##Comparación entre IaaS, PaaS y SaaS

Para el caso de Wanderbricks se comparan los tres modelos considerando control, tiempo de implementación, esfuerzo operativo, costos, escalabilidad y gobierno.

| Criterio                           | IaaS                                                                                                       | PaaS                                                                                                                        | SaaS                                                                                                   |
| ---------------------------------- | ---------------------------------------------------------------------------------------------------------- | --------------------------------------------------------------------------------------------------------------------------- | ------------------------------------------------------------------------------------------------------ |
| **Control**                        | Alto. El equipo administra máquinas, sistema operativo y software.                                         | Medio-alto. Se controla la configuración y los procesos, mientras el proveedor administra gran parte de la infraestructura. | Bajo. La aplicación está principalmente administrada por el proveedor.                                 |
| **Tiempo para obtener resultados** | Mayor, debido a la configuración inicial de infraestructura y software.                                    | Menor, porque gran parte del entorno de procesamiento ya está disponible.                                                   | Generalmente menor para utilizar una aplicación terminada, aunque depende de las necesidades del caso. |
| **Esfuerzo operativo**             | Alto. Requiere administrar servidores, SO, red, almacenamiento y software.                                 | Medio. El proveedor administra buena parte de la infraestructura.                                                           | Bajo. La mayor parte de la operación está a cargo del proveedor.                                       |
| **Costos**                         | Dependen de las máquinas, almacenamiento, red, licencias y operación necesarias.                           | Se concentran principalmente en el consumo de los servicios utilizados.                                                     | Generalmente se paga por suscripción, usuarios o consumo del servicio.                                 |
| **Escalabilidad**                  | Requiere dimensionar y ampliar infraestructura.                                                            | Permite aprovechar mecanismos de escalamiento proporcionados por la plataforma.                                             | Depende de las capacidades y límites establecidos por el proveedor.                                    |
| **Gobierno**                       | El equipo tiene amplio control, pero debe implementar directamente los mecanismos de seguridad y gobierno. | Permite utilizar mecanismos integrados de catálogos, permisos, auditoría y trazabilidad.                                    | El gobierno depende principalmente de las capacidades ofrecidas por el proveedor.                      |


## Conclusión

Para el caso desarrollado en Wanderbricks, el modelo PaaS resulta adecuado para las necesidades del proyecto, debido a que permite concentrar el trabajo en la ingeniería y análisis de datos sin asumir la administración completa de servidores y sistemas operativos.

La arquitectura desarrollada utiliza Bronze, Silver y Gold para organizar el procesamiento de los datos, mientras que las capacidades de catálogo, permisos, lineage y Jobs permiten incorporar elementos de gobierno y automatización dentro del mismo entorno.

IaaS proporciona mayor control sobre la infraestructura, pero implica asumir tareas adicionales de instalación, mantenimiento, seguridad, actualización, monitoreo y escalamiento. SaaS reduce aún más la administración técnica, pero no está orientado de la misma manera a construir y administrar una plataforma de procesamiento de datos personalizada como la desarrollada en este proyecto.

Por estas razones, para este caso académico y para una solución que requiere procesamiento, transformación, gobierno, automatización y trazabilidad de datos, PaaS permite mantener un equilibrio entre control, velocidad de implementación y esfuerzo operativo.

## Proyecto final: procesamiento distribuido y cierre del caso

aqui verificamos el estado de nuestras tablas en gold con las que posteriormente comenzaremos a trabajar
realizamos las consultas correspondientes para poder iniciar sin realizar cambios extremos en nuestra arquitectura

In [0]:
spark.sql("SELECT current_catalog(), current_schema()").show()

+-----------------+----------------+
|current_catalog()|current_schema()|
+-----------------+----------------+
|        workspace|         default|
+-----------------+----------------+



In [0]:
spark.sql("SHOW CATALOGS").show()

+--------------------+
|             catalog|
+--------------------+
|             samples|
|              system|
|wanderbricks_anal...|
|           workspace|
+--------------------+



In [0]:
spark.sql("SHOW SCHEMAS").show()

+------------------+
|      databaseName|
+------------------+
|           default|
|information_schema|
+------------------+



In [0]:
spark.sql("SHOW SCHEMAS IN wanderbricks_analitica").show(truncate=False)

+------------------+
|databaseName      |
+------------------+
|bronze            |
|default           |
|gold              |
|information_schema|
|silver            |
+------------------+



In [0]:
spark.sql("""
SHOW TABLES IN wanderbricks_analitica.gold
""").show(truncate=False)

+--------+-------------------------+-----------+
|database|tableName                |isTemporary|
+--------+-------------------------+-----------+
|gold    |demo_acid                |false      |
|gold    |resumen_metodos_pago     |false      |
|gold    |resumen_rendimiento_hosts|false      |
|gold    |resumen_reservas_destino |false      |
|gold    |resumen_usuarios         |false      |
+--------+-------------------------+-----------+



In [0]:
spark.sql("""
SHOW TABLES IN wanderbricks_analitica.gold
""").show(truncate=False)

+--------+-------------------------+-----------+
|database|tableName                |isTemporary|
+--------+-------------------------+-----------+
|gold    |demo_acid                |false      |
|gold    |resumen_metodos_pago     |false      |
|gold    |resumen_rendimiento_hosts|false      |
|gold    |resumen_reservas_destino |false      |
|gold    |resumen_usuarios         |false      |
+--------+-------------------------+-----------+



In [0]:
spark.sql("""
SELECT *
FROM wanderbricks_analitica.gold.resumen_reservas_destino
LIMIT 10
""").show(truncate=False)

+-------------+--------------+----------------+----------------+-----------------+
|destination  |total_reservas|ingresos_totales|ingreso_promedio|categoria_destino|
+-------------+--------------+----------------+----------------+-----------------+
|Beijing      |15552         |9395956.8000    |604.16388889    |Alta demanda     |
|Cairo        |222480        |121236341.8032  |544.93141767    |Alta demanda     |
|Sardinia     |378000        |211687421.7168  |560.01963417    |Alta demanda     |
|Berlin       |480600        |266840924.4000  |555.22456180    |Alta demanda     |
|San Francisco|540432        |295162410.2616  |546.16012794    |Alta demanda     |
|Osaka        |661392        |366778031.0616  |554.55468325    |Alta demanda     |
|Banff        |10800         |5945056.5600    |550.46820000    |Alta demanda     |
|Mallorca     |1401192       |770178281.0616  |549.65934794    |Alta demanda     |
|Abu Dhabi    |953424        |528326919.4248  |554.13637524    |Alta demanda     |
|Kit

## ahor contamos nuestras tres capaz


In [0]:
print("BRONZE")
spark.sql("""
SELECT COUNT(*) AS registros
FROM wanderbricks_analitica.bronze.bookings
""").show()

print("SILVER")
spark.sql("""
SELECT COUNT(*) AS registros
FROM wanderbricks_analitica.silver.bookings
""").show()

print("GOLD")
spark.sql("""
SELECT COUNT(*) AS registros
FROM wanderbricks_analitica.gold.resumen_reservas_destino
""").show()

BRONZE
+---------+
|registros|
+---------+
|   216741|
+---------+

SILVER
+---------+
|registros|
+---------+
|   433476|
+---------+

GOLD
+---------+
|registros|
+---------+
|       41|
+---------+



In [0]:
spark.sql("""
DESCRIBE wanderbricks_analitica.bronze.bookings
""").show(truncate=False)

+------------+---------+-------+
|col_name    |data_type|comment|
+------------+---------+-------+
|booking_id  |bigint   |NULL   |
|user_id     |bigint   |NULL   |
|property_id |bigint   |NULL   |
|check_in    |date     |NULL   |
|check_out   |date     |NULL   |
|guests_count|int      |NULL   |
|total_amount|float    |NULL   |
|status      |string   |NULL   |
|created_at  |timestamp|NULL   |
|updated_at  |timestamp|NULL   |
+------------+---------+-------+



In [0]:
spark.sql("""
DESCRIBE wanderbricks_analitica.silver.bookings
""").show(truncate=False)

+------------+-------------+-------+
|col_name    |data_type    |comment|
+------------+-------------+-------+
|booking_id  |bigint       |NULL   |
|user_id     |bigint       |NULL   |
|property_id |bigint       |NULL   |
|check_in    |date         |NULL   |
|check_out   |date         |NULL   |
|guests_count|int          |NULL   |
|total_amount|decimal(15,4)|NULL   |
|status      |string       |NULL   |
|created_at  |timestamp    |NULL   |
|updated_at  |timestamp    |NULL   |
+------------+-------------+-------+



In [0]:
print("BRONZE")
spark.sql("""
SELECT
    MIN(check_in) AS primera_fecha,
    MAX(check_in) AS ultima_fecha
FROM wanderbricks_analitica.bronze.bookings
""").show()

print("SILVER")
spark.sql("""
SELECT
    MIN(check_in) AS primera_fecha,
    MAX(check_in) AS ultima_fecha
FROM wanderbricks_analitica.silver.bookings
""").show()

BRONZE
+-------------+------------+
|primera_fecha|ultima_fecha|
+-------------+------------+
|   2022-12-29|  2025-07-30|
+-------------+------------+

SILVER
+-------------+------------+
|primera_fecha|ultima_fecha|
+-------------+------------+
|   2022-12-29|  2025-07-30|
+-------------+------------+



In [0]:
print("BRONZE")
spark.sql("""
SELECT
    COUNT(*) AS filas,
    COUNT(DISTINCT booking_id) AS bookings_unicos
FROM wanderbricks_analitica.bronze.bookings
""").show()

print("SILVER")
spark.sql("""
SELECT
    COUNT(*) AS filas,
    COUNT(DISTINCT booking_id) AS bookings_unicos
FROM wanderbricks_analitica.silver.bookings
""").show()

BRONZE
+------+---------------+
| filas|bookings_unicos|
+------+---------------+
|216741|          72247|
+------+---------------+

SILVER
+------+---------------+
| filas|bookings_unicos|
+------+---------------+
|433476|          72246|
+------+---------------+



In [0]:
spark.sql("""
SELECT
    booking_id,
    COUNT(*) AS cantidad_filas
FROM wanderbricks_analitica.silver.bookings
GROUP BY booking_id
ORDER BY cantidad_filas DESC
LIMIT 20
""").show()

+----------+--------------+
|booking_id|cantidad_filas|
+----------+--------------+
|     18543|             6|
|     18469|             6|
|     18499|             6|
|     18504|             6|
|     18383|             6|
|     18456|             6|
|     18375|             6|
|     18493|             6|
|     18370|             6|
|     18367|             6|
|     18170|             6|
|     18348|             6|
|     18327|             6|
|     18405|             6|
|     18159|             6|
|     18366|             6|
|     18288|             6|
|     18478|             6|
|     18223|             6|
|     18550|             6|
+----------+--------------+



In [0]:
spark.sql("""
SELECT
    cantidad_filas,
    COUNT(*) AS cantidad_reservas
FROM (
    SELECT
        booking_id,
        COUNT(*) AS cantidad_filas
    FROM wanderbricks_analitica.silver.bookings
    GROUP BY booking_id
)
GROUP BY cantidad_filas
ORDER BY cantidad_filas
""").show()

+--------------+-----------------+
|cantidad_filas|cantidad_reservas|
+--------------+-----------------+
|             6|            72246|
+--------------+-----------------+



In [0]:
spark.sql("""
SELECT *
FROM wanderbricks_analitica.silver.bookings
WHERE booking_id = 18543
""").show(truncate=False)

+----------+-------+-----------+----------+----------+------------+------------+-------+-------------------+-------------------+
|booking_id|user_id|property_id|check_in  |check_out |guests_count|total_amount|status |created_at         |updated_at         |
+----------+-------+-----------+----------+----------+------------+------------+-------+-------------------+-------------------+
|18543     |28473  |10984      |2025-06-01|2025-06-06|1           |740.4300    |pending|2025-05-24 03:26:05|2025-05-27 12:08:17|
|18543     |28473  |10984      |2025-06-01|2025-06-06|1           |740.4300    |pending|2025-05-24 03:26:05|2025-05-27 12:08:17|
|18543     |28473  |10984      |2025-06-01|2025-06-06|1           |740.4300    |pending|2025-05-24 03:26:05|2025-05-27 12:08:17|
|18543     |28473  |10984      |2025-06-01|2025-06-06|1           |740.4300    |pending|2025-05-24 03:26:05|2025-05-27 12:08:17|
|18543     |28473  |10984      |2025-06-01|2025-06-06|1           |740.4300    |pending|2025-05-2

In [0]:
for tabla in ["bookings", "properties", "destinations", "users"]:
    print(f"\n===== {tabla.upper()} =====")
    spark.sql(f"""
        DESCRIBE wanderbricks_analitica.silver.{tabla}
    """).show(truncate=False)


===== BOOKINGS =====
+------------+-------------+-------+
|col_name    |data_type    |comment|
+------------+-------------+-------+
|booking_id  |bigint       |NULL   |
|user_id     |bigint       |NULL   |
|property_id |bigint       |NULL   |
|check_in    |date         |NULL   |
|check_out   |date         |NULL   |
|guests_count|int          |NULL   |
|total_amount|decimal(15,4)|NULL   |
|status      |string       |NULL   |
|created_at  |timestamp    |NULL   |
|updated_at  |timestamp    |NULL   |
+------------+-------------+-------+


===== PROPERTIES =====
+------------------+-------------+-------+
|col_name          |data_type    |comment|
+------------------+-------------+-------+
|property_id       |bigint       |NULL   |
|host_id           |bigint       |NULL   |
|destination_id    |bigint       |NULL   |
|title             |string       |NULL   |
|description       |string       |NULL   |
|base_price        |decimal(15,4)|NULL   |
|property_type     |string       |NULL   |
|max_

## despues de conocer el resultado y el estado de nuestas tablas preparamos el Json para inicial


In [0]:
from time import perf_counter

bookings = spark.table("wanderbricks_analitica.silver.bookings")
properties = spark.table("wanderbricks_analitica.silver.properties")

join_inicial = (
    bookings
    .join(
        properties,
        bookings.property_id == properties.property_id,
        "inner"
    )
    .select(
        bookings.booking_id,
        bookings.user_id,
        bookings.property_id,
        bookings.total_amount,
        properties.host_id,
        properties.destination_id,
        properties.property_type
    )
)

join_inicial.explain("formatted")

== Physical Plan ==
AdaptiveSparkPlan (10)
+- == Initial Plan ==
   PhotonResultStage (9)
   +- PhotonColumnarToRow (8)
      +- PhotonProject (7)
         +- PhotonBroadcastHashJoin Inner (6)
            :- PhotonScan parquet wanderbricks_analitica.silver.bookings (1)
            +- PhotonShuffleExchangeSource (5)
               +- PhotonShuffleMapStage (4)
                  +- PhotonShuffleExchangeSink (3)
                     +- PhotonScan parquet wanderbricks_analitica.silver.properties (2)


(1) PhotonScan parquet wanderbricks_analitica.silver.bookings
Output [4]: [booking_id#11431L, user_id#11432L, property_id#11433L, total_amount#11437]
Location: PreparedDeltaFileIndex [s3://dbstorage-prod-8yhlq/uc/612cc16c-5fe0-45b5-8ca4-0e8a4db1dcbc/84fdfc38-fd89-4539-a946-e891148c5c51/__unitystorage/catalogs/1fe7c40a-50f2-402d-b89f-adb9a4e0b0ad/tables/2a33af71-5664-4100-9f34-e7941046107a]
OptionalDataFilters: [hashedrelationcontains(property_id#11433L)]
ReadSchema: struct<booking_id:bigint,us

In [0]:
from time import perf_counter
import statistics

# Una ejecución de calentamiento (NO se registra)
join_inicial.count()

# Tres mediciones
tiempos_iniciales = []

for i in range(3):
    inicio = perf_counter()
    
    resultado = join_inicial.count()
    
    fin = perf_counter()
    tiempo = fin - inicio
    tiempos_iniciales.append(tiempo)
    
    print(f"Ejecución {i+1}: {tiempo:.4f} segundos")

mediana_inicial = statistics.median(tiempos_iniciales)

print(f"\nMediana inicial: {mediana_inicial:.4f} segundos")
print(f"Filas resultantes: {resultado}")

Ejecución 1: 1.5652 segundos
Ejecución 2: 1.1317 segundos
Ejecución 3: 1.2100 segundos

Mediana inicial: 1.2100 segundos
Filas resultantes: 2600856


In [0]:
bookings_confirmadas = (
    bookings
    .filter(bookings.status == "confirmed")
)

join_optimizado = (
    bookings_confirmadas
    .join(
        properties,
        bookings_confirmadas.property_id == properties.property_id,
        "inner"
    )
    .select(
        bookings_confirmadas.booking_id,
        bookings_confirmadas.user_id,
        bookings_confirmadas.property_id,
        bookings_confirmadas.total_amount,
        properties.host_id,
        properties.destination_id,
        properties.property_type
    )
)

join_optimizado.explain("formatted")

== Physical Plan ==
AdaptiveSparkPlan (11)
+- == Initial Plan ==
   PhotonResultStage (10)
   +- PhotonColumnarToRow (9)
      +- PhotonProject (8)
         +- PhotonBroadcastHashJoin Inner (7)
            :- PhotonShuffleExchangeSource (5)
            :  +- PhotonShuffleMapStage (4)
            :     +- PhotonShuffleExchangeSink (3)
            :        +- PhotonProject (2)
            :           +- PhotonScan parquet wanderbricks_analitica.silver.bookings (1)
            +- PhotonScan parquet wanderbricks_analitica.silver.properties (6)


(1) PhotonScan parquet wanderbricks_analitica.silver.bookings
Output [5]: [booking_id#11870L, user_id#11871L, property_id#11872L, total_amount#11876, status#11877]
DictionaryFilters: [(status#11877 = confirmed)]
Location: PreparedDeltaFileIndex [s3://dbstorage-prod-8yhlq/uc/612cc16c-5fe0-45b5-8ca4-0e8a4db1dcbc/84fdfc38-fd89-4539-a946-e891148c5c51/__unitystorage/catalogs/1fe7c40a-50f2-402d-b89f-adb9a4e0b0ad/tables/2a33af71-5664-4100-9f34-e7941046107

In [0]:
# Una ejecución de calentamiento (NO se registra)
join_optimizado.count()

# Tres mediciones
tiempos_optimizados = []

for i in range(3):
    inicio = perf_counter()
    
    resultado_optimizado = join_optimizado.count()
    
    fin = perf_counter()
    tiempo = fin - inicio
    tiempos_optimizados.append(tiempo)
    
    print(f"Ejecución {i+1}: {tiempo:.4f} segundos")

mediana_optimizada = statistics.median(tiempos_optimizados)

print(f"\nMediana optimizada: {mediana_optimizada:.4f} segundos")
print(f"Filas resultantes: {resultado_optimizado}")

Ejecución 1: 1.1348 segundos
Ejecución 2: 1.1316 segundos
Ejecución 3: 1.6318 segundos

Mediana optimizada: 1.1348 segundos
Filas resultantes: 646128


In [0]:
spark.sql("""
SELECT
    cantidad_filas,
    COUNT(*) AS cantidad_reservas
FROM (
    SELECT
        booking_id,
        COUNT(*) AS cantidad_filas
    FROM wanderbricks_analitica.bronze.bookings
    GROUP BY booking_id
)
GROUP BY cantidad_filas
ORDER BY cantidad_filas
""").show()

+--------------+-----------------+
|cantidad_filas|cantidad_reservas|
+--------------+-----------------+
|             3|            72247|
+--------------+-----------------+



Bronze: 216.741 registros correspondientes a 72.247 reservas únicas. Cada booking_id aparece tres veces en esta capa, por lo que la diferencia entre registros físicos y reservas únicas corresponde a la estructura de los datos de origen.

Silver: 433.476 registros correspondientes a 72.246 reservas únicas. Cada reserva aparece seis veces. Frente a Bronze se conserva prácticamente el mismo conjunto de reservas, con la diferencia de un booking_id, mientras que aumenta la cantidad de filas por reserva. Esto indica que la transformación de Silver trabaja con una representación de mayor cardinalidad y no debe interpretarse simplemente como una eliminación de duplicados.

Gold: 41 registros en resumen_reservas_destino, donde la información operacional se transforma en una agregación por destino. En esta capa cada fila representa un destino y contiene métricas calculadas como número de reservas, ingresos totales e ingreso promedio, facilitando directamente su interpretación desde el punto de vista del negocio.

## Cuándo esta optimización podría ser contraproducente.


Aplicar filtros antes de un JOIN es conveniente cuando el filtro elimina una cantidad significativa de registros y la consulta realmente necesita solo ese subconjunto. Puede ser contraproducente si el filtro elimina muy pocos registros, si el negocio necesita conservar todos los estados o si el filtrado implica una operación costosa que no compensa la reducción posterior.

In [0]:
spark.sql("""
SELECT
    destination,
    total_reservas,
    ingresos_totales,
    ingreso_promedio,
    categoria_destino
FROM wanderbricks_analitica.gold.resumen_reservas_destino
ORDER BY total_reservas DESC
""").show(41, truncate=False)

+-------------+--------------+----------------+----------------+-----------------+
|destination  |total_reservas|ingresos_totales|ingreso_promedio|categoria_destino|
+-------------+--------------+----------------+----------------+-----------------+
|Phuket       |1575072       |876571102.8000  |556.52763988    |Alta demanda     |
|Gold Coast   |1405512       |779780334.2832  |554.80161982    |Alta demanda     |
|Mallorca     |1401192       |770178281.0616  |549.65934794    |Alta demanda     |
|Paris        |1219536       |680010640.6032  |557.59784098    |Alta demanda     |
|Abu Dhabi    |953424        |528326919.4248  |554.13637524    |Alta demanda     |
|Tokyo        |778248        |432655274.8584  |555.93496528    |Alta demanda     |
|Dubai        |669600        |367655593.6800  |549.06749355    |Alta demanda     |
|Osaka        |661392        |366778031.0616  |554.55468325    |Alta demanda     |
|Singapore    |648864        |360565536.2400  |555.68738016    |Alta demanda     |
|New

## Recorrido de los datos: Bronze → Silver → Gold

El pipeline de Wanderbricks utiliza una arquitectura por capas para llevar los datos desde su representación inicial hasta una estructura orientada al consumo analítico.

**Bronze:** la tabla `bookings` contiene 216.741 registros correspondientes a 72.247 `booking_id` únicos. Cada reserva aparece tres veces en esta capa. El rango de `check_in` va desde el 29 de diciembre de 2022 hasta el 30 de julio de 2025.

**Silver:** la tabla `bookings` contiene 433.476 registros correspondientes a 72.246 `booking_id` únicos. Cada reserva presente en esta capa aparece seis veces. El rango de fechas se mantiene entre el 29 de diciembre de 2022 y el 30 de julio de 2025. Frente a Bronze se conserva prácticamente el mismo conjunto de reservas, con una diferencia de un `booking_id`, mientras que aumenta la cardinalidad de las filas. Por ello, la diferencia no se interpreta simplemente como eliminación de duplicados, sino como resultado de la transformación aplicada en esta capa.

**Gold:** la tabla `resumen_reservas_destino` contiene 41 registros, donde cada fila representa un destino con información agregada para el negocio: total de reservas, ingresos totales, ingreso promedio y categoría de demanda. Los resultados están ordenados de mayor a menor número de reservas, permitiendo identificar directamente los destinos con mayor volumen de actividad.

En conjunto, el recorrido transforma los datos desde una estructura operacional de mayor detalle en Bronze y Silver hacia una representación agregada y orientada a la toma de decisiones en Gold.

In [0]:
spark.sql("""
SELECT
    COUNT(*) AS filas_gold,
    COUNT(DISTINCT destination) AS destinos_unicos
FROM wanderbricks_analitica.gold.resumen_reservas_destino
""").show()

+----------+---------------+
|filas_gold|destinos_unicos|
+----------+---------------+
|        41|             41|
+----------+---------------+



In [0]:
spark.sql("""
SELECT
    destination,
    COUNT(*) AS cantidad_filas,
    SUM(total_reservas) AS reservas_acumuladas,
    SUM(ingresos_totales) AS ingresos_acumulados
FROM wanderbricks_analitica.gold.resumen_reservas_destino
GROUP BY destination
ORDER BY reservas_acumuladas DESC
""").show(truncate=False)

+-------------+--------------+-------------------+-------------------+
|destination  |cantidad_filas|reservas_acumuladas|ingresos_acumulados|
+-------------+--------------+-------------------+-------------------+
|Phuket       |1             |1575072            |876571102.8000     |
|Gold Coast   |1             |1405512            |779780334.2832     |
|Mallorca     |1             |1401192            |770178281.0616     |
|Paris        |1             |1219536            |680010640.6032     |
|Abu Dhabi    |1             |953424             |528326919.4248     |
|Tokyo        |1             |778248             |432655274.8584     |
|Dubai        |1             |669600             |367655593.6800     |
|Osaka        |1             |661392             |366778031.0616     |
|Singapore    |1             |648864             |360565536.2400     |
|New York     |1             |632664             |353260284.4368     |
|Barcelona    |1             |601560             |335069449.1784     |
|San F

In [0]:
spark.sql("""
SELECT
    COUNT(*) AS filas_gold,
    COUNT(DISTINCT TRIM(destination)) AS destinos_unicos
FROM wanderbricks_analitica.gold.resumen_reservas_destino
""").show()

+----------+---------------+
|filas_gold|destinos_unicos|
+----------+---------------+
|        41|             41|
+----------+---------------+



## Pregunta de negocio respondida en Gold

**¿Qué destinos concentran la mayor cantidad de reservas y qué ingresos generan?**

La tabla `resumen_reservas_destino` responde esta pregunta mediante una agregación por destino. Para cada destino se presentan el total de reservas, los ingresos totales, el ingreso promedio y una categoría de demanda. Los resultados se ordenan de mayor a menor cantidad de reservas, de modo que la información de mayor relevancia para el negocio aparece primero.

La capa Gold contiene 41 filas correspondientes a 41 destinos únicos, por lo que cada registro representa un destino diferente y sus métricas agregadas.

## Optimización de la operación distribuida

La operación seleccionada fue un JOIN entre las tablas `silver.bookings` y `silver.properties`, utilizando `property_id` como clave de relación.

### Plan inicial

El plan físico mostró un `PhotonBroadcastHashJoin`. Photon identificó que la tabla `properties` podía utilizarse mediante broadcast, evitando un `SortMergeJoin` convencional. Por esta razón, el broadcast no se utilizó como técnica adicional de optimización, ya que el motor ya lo estaba aplicando automáticamente.

La medición inicial se realizó después de una ejecución de calentamiento y mediante tres ejecuciones independientes:

- Ejecución 1: 1,5652 segundos
- Ejecución 2: 1,1317 segundos
- Ejecución 3: 1,2100 segundos
- **Mediana: 1,2100 segundos**
- Filas resultantes: 2.600.856

### Optimización aplicada

Se aplicó un filtro sobre `status = 'confirmed'` antes de realizar el JOIN. El plan físico resultante mostró que el filtro fue aplicado durante la lectura de `bookings`, mediante `DictionaryFilters` y `RequiredDataFilters`, antes de completar el JOIN.

La medición optimizada, utilizando nuevamente una ejecución de calentamiento y tres mediciones, fue:

- Ejecución 1: 1,1348 segundos
- Ejecución 2: 1,1316 segundos
- Ejecución 3: 1,6318 segundos
- **Mediana: 1,1348 segundos**
- Filas resultantes: 646.128

La mediana observada pasó de 1,2100 a 1,1348 segundos, una reducción aproximada del 6,2 %. Al mismo tiempo, el resultado pasó de 2.600.856 a 646.128 filas, una reducción aproximada del 75,2 %.

### Interpretación

La principal diferencia observable en el plan es que el filtro `status = 'confirmed'` se aplica directamente durante la lectura de `bookings`. Esto permite reducir los datos que continúan hacia las siguientes etapas de procesamiento y hacia el JOIN. El `PhotonBroadcastHashJoin` permanece presente, por lo que la optimización no consistió en cambiar el tipo de JOIN, sino en reducir previamente el volumen de datos que participa en la operación.

La reducción del tiempo de ejecución debe interpretarse junto con la variabilidad observada entre las ejecuciones. La mediana disminuyó aproximadamente un 6,2 %, mientras que el volumen de resultados se redujo considerablemente. Por ello, la evidencia más clara de la optimización está en el cambio del plan y en la reducción del volumen procesado, además de la disminución observada en la mediana.

Esta estrategia puede ser contraproducente cuando el filtro elimina muy pocos registros, cuando la consulta necesita todos los estados de las reservas o cuando el costo adicional de aplicar el filtro supera el beneficio obtenido por reducir los datos posteriores.

## “Una empresa recibe datos de su portal web, de Facebook, de Instagram, de TikTok y de WhatsApp Business. ¿Bajo qué paradigma, con qué herramientas y con qué arquitectura debería analizarlos?”

Una empresa que recibe datos de su portal web, Facebook, Instagram, TikTok y WhatsApp Business debería analizarlos bajo un paradigma de procesamiento distribuido y orientado a datos, utilizando herramientas capaces de integrar diferentes fuentes y procesar grandes volúmenes de información. La experiencia desarrollada con Wanderbricks muestra que una plataforma como Databricks, apoyada en Apache Spark y Photon, permite realizar transformaciones y consultas de forma distribuida, mientras que Delta Lake aporta almacenamiento estructurado y confiable. Esta combinación permite recibir datos de diferentes fuentes, almacenarlos inicialmente en una capa Bronze, limpiarlos y transformarlos en Silver y finalmente construir información agregada y orientada al negocio en Gold.

La arquitectura recomendada sería, por tanto, una arquitectura por capas Bronze, Silver y Gold, en la que cada nivel tenga una responsabilidad definida. En Bronze se conservarían los datos provenientes de las diferentes plataformas; en Silver se realizarían procesos de limpieza, estandarización, validación y transformación; y en Gold se construirían tablas agregadas con métricas directamente utilizables por el negocio. Sobre esta última capa podrían implementarse herramientas de consumo como consultas SQL, dashboards o soluciones de analítica. Esta arquitectura permite separar la ingesta, el procesamiento y el consumo, facilita la trazabilidad de los datos y permite optimizar las operaciones distribuidas, como se comprobó durante el proyecto al comparar los planes de ejecución y medir el comportamiento de un JOIN antes y después de aplicar una optimización.


## Conclusiones finales

El desarrollo del pipeline permitió comprobar la utilidad de una arquitectura por capas para transformar datos operacionales en información orientada al consumo analítico. La separación entre Bronze, Silver y Gold facilitó organizar el procesamiento y construir una tabla Gold con métricas directamente interpretables por el negocio. También fue posible analizar una operación distribuida mediante sus planes de ejecución y mediciones reales. La optimización aplicada al JOIN permitió observar cómo un filtro puede ser incorporado durante la lectura de los datos y reducir el volumen que continúa hacia las siguientes etapas del procesamiento.

No todas las situaciones pueden resolverse simplemente aplicando una técnica de optimización. En este caso, Photon ya utilizaba automáticamente un `BroadcastHashJoin`, por lo que aplicar broadcast manualmente no habría representado una mejora adicional evidente. La medición también mostró variabilidad entre ejecuciones, por lo que los resultados deben analizarse mediante varias mediciones y no a partir de una sola ejecución. Si se desarrollara nuevamente el proyecto, sería conveniente controlar con mayor detalle la calidad y duplicidad de los datos desde las primeras etapas y evaluar otras estrategias de optimización sobre operaciones de mayor costo. En conjunto, el ejercicio permitió relacionar la arquitectura de datos, el procesamiento distribuido, los planes de ejecución y la medición de rendimiento con una necesidad concreta de negocio.
